# Programmation linéaire fortement polynomiale : le neuvième problème de Smale

**Série Complexity — accrétion 07d** · Public : **Recherche** · Hommage fondateur : **OpenAI, *Strongly polynomial linear programming* (prépublication du 24 novembre 2026), qui répond au neuvième problème de Smale**

**Navigation** : [<< Complexity-07c — Le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) | [Index de la série](README.md) | [Complexity-08 — Wolfram, règle 30 aléatoire, règle 110 universelle →](Complexity-08-WolframRule30Aleatoire-Rule110Universelle-Python.ipynb)

Ce carnet est une **accrétion de [Complexity-07 — Complexité de Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb)**, et il reprend sa question centrale — *que mesure exactement la taille d'une entrée ?* — sur le problème d'optimisation le plus utilisé au monde. La base 07 montre qu'une même fonction peut être décrite par des programmes de longueurs très différentes ; l'accrétion **07b** parcourt les échelles de grandeur ; **07c** dresse le bilan de la vague de résultats d'octobre 2026 et dit d'elle-même, à propos de ces annonces : « il renvoie, il ne refait pas ». **07d** fait le contraire : elle prend un résultat de la vague suivante — celle du 24 novembre 2026 — et le travaille au fond, jusqu'à comparer ce que le manuscrit affirme et ce que la formalisation machine démontre.

**Prérequis ajoutés par cette accrétion** (ils ne sont pas nécessaires aux positions précédentes) : dualité d'un programme linéaire et complémentarité ; notion de certificat d'infaisabilité (alternative de Farkas) ; arithmétique rationnelle exacte avec `fractions.Fraction` ; lecture d'un énoncé Lean 4. La notion de taille d'entrée vient de [Complexity-01 — Compter des pas](Complexity-01-StepCounting-Python.ipynb).

## Objectifs d'apprentissage

1. Distinguer les deux manières de compter le coût d'un algorithme de programmation linéaire — en **longueur binaire de l'entrée**, et en **nombre de variables et de contraintes** — et voir pourquoi la seconde était ouverte.
2. Exécuter un simplexe exact et **compter ses pivots** ; vérifier qu'un programme de deux variables et deux contraintes par dimension peut demander un nombre de pivots exponentiel en la dimension (Klee–Minty), alors que l'optimum est à une arête du point de départ.
3. Mesurer qu'à **dimensions constantes** le nombre de coupes d'un ellipsoïde croît avec la longueur de l'entrée, tandis que le simplexe ne bouge pas.
4. Vérifier exhaustivement l'alternative de Farkas — et sa **borne de support** — sur de petits systèmes.
5. Lire l'écart entre l'énoncé du manuscrit (données réelles arbitraires, complémentarité stricte) et les énoncés formalisés en Lean (rationnels, sorties sans complémentarité stricte), et savoir **laquelle des deux formulations une phrase peut citer**.

## Durée estimée

Environ deux heures de travail : lire le carnet, l'exécuter et traiter les trois exercices ; la lecture seule du texte peut se faire en une heure.

## Conventions de lecture

Chaque affirmation de ce carnet est **Mesurée** — produite par une cellule de ce carnet, en arithmétique rationnelle exacte sauf mention explicite du flottant — ou **Citée** — attribuée à une source nominative, lue dans sa copie archivée. Aucune valeur n'est reprise de mémoire.

Les sources primaires sont archivées au gisement partagé, hors dépôt :

- `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - OpenAI - Strongly polynomial linear programming (openai-math preprint, 24 Nov 2026).pdf`
- `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - OpenAI - Superpolynomial graph diameter of simple polytopes (openai-math preprint, 24 Nov 2026).pdf`
- `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\1972 - Klee Minty - How Good Is the Simplex Algorithm (Inequalities III pp. 159-175, scan sophie.huiberts.me).pdf`
- `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\1998 - Smale - Mathematical Problems for the Next Century (Math. Intelligencer 20-2 pp. 7-15, seconde version du 7 aout 1998, copie uni-passau.de).pdf`

La formalisation machine accompagne le premier texte : le dépôt `openai/math` contient le développement Lean 4 (`lean/OAI/Optimization/LinearProgramming/`) et sa notice (`overview.tex`). Ce carnet en cite les énoncés avec leur signature exacte.

Les citations sont données dans les mots de leur source. Seule la typographie mathématique est transposée : une formule affichée est rendue sur une ligne (exposants, indices, sommations), une matrice en notation à crochets, un exposant détaché de son objet par l'extraction est recomposé sur la même ligne, et toute coupe est marquée `[...]`.

## 1. Le problème en une page

**Optimiser** un programme linéaire, c'est calculer le maximum de `c·x` sous `A x ≤ b`. Les données sont trois objets : la matrice `A ∈ R^{m×n}`, le vecteur `b ∈ R^m`, le vecteur `c ∈ R^n`.

Deux façons de mesurer le travail :

| Ce qu'on compte | Ce que le résultat garantit |
|---|---|
| la **longueur binaire** `L` de l'écriture de `A, b, c` | un nombre d'opérations polynomial en `L` — c'est ce que donne l'ellipsoïde de Khachiyan (1979) |
| les **dimensions** `m` et `n` seules | un nombre d'opérations polynomial en `m + n`, indépendant des coefficients — c'est la classe dite **fortement polynomiale** |

Les deux ne coïncident pas, parce que deux programmes de mêmes dimensions peuvent avoir des écritures de longueurs très différentes : écrire `1` ou `2^{1000}` ne change ni `m` ni `n`. Le texte cité l'énonce ainsi :

> Two linear programs with the same numbers of variables and constraints can have very different input lengths. Megiddo [104, 105] asked whether those two dimensions alone could bound the number of arithmetic operations. Tarjan raised the question for real data [135, pp. 1620–1621], and Smale made it his ninth problem [130]. We give an affirmative answer.

> **Theorem 1 (Strongly polynomial linear programming).** For arbitrary real `A`, `b`, `c`, a deterministic strongly polynomial algorithm returns one of the following certificates for `max{c⊤x : Ax ≤ b}`.
> 1. **Finite optimum.** A strictly complementary primal–dual pair `x, y` with `s = b − Ax ≥ 0`, `y ≥ 0`, `A⊤y = c`, `s⊤y = 0`, `s + y > 0`.
> 2. **Unbounded objective.** A feasible point `x` and an improving ray `r` with `Ar ≤ 0` and `c⊤r > 0`.
> 3. **Infeasibility.** A vector `y ≥ 0` with `A⊤y = 0`, `b⊤y < 0`, and `|supp y| ≤ 1 + rank A`.

Trois choses à retenir de cet énoncé, parce que tout le carnet en dépend.

1. Il y a **trois sorties possibles**, et chacune est un *certificat* vérifiable : un couple optimal, un rayon, un vecteur de Farkas. Un algorithme qui répondrait « je ne sais pas » ne satisferait pas le théorème.
2. La complémentarité demandée est **stricte** (`s + y > 0`), pas seulement faible (`s^T y = 0`) : pour chaque paire (écart, multiplicateur), au moins un des deux termes est non nul.
3. Le certificat d'infaisabilité porte une **borne de support** : au plus `1 + rang A` coordonnées non nulles, donc une preuve courte même quand il y a beaucoup de contraintes.

Le modèle de calcul est précisé juste après l'énoncé :

> A strongly polynomial algorithm takes polynomially many steps in `m, n`, counting exact scalar arithmetic, comparisons and control work. On rational input every intermediate number has bit length polynomial in the full binary input length.

Autrement dit : le **nombre de pas** ne dépend que de `m + n` ; la **taille des nombres** reste, elle, polynomiale en la longueur de l'entrée. C'est la combinaison des deux qui rend l'algorithme utilisable — un algorithme dont les nombres doubleraient de taille à chaque pas ne serait pas fortement polynomial au sens utile.

La question elle-même remonte à la liste publiée par **Steve Smale** en 1998, où elle porte le numéro neuf. La copie archivée permet de la citer dans ses mots :

> Problem 9: The linear programming problem. Is there a polynomial time algorithm over the real numbers which decides the feasibility of the linear system of inequalities Ax ≥b?

Le modèle est celui d'une machine à nombres réels, et le temps s'y compte en opérations arithmétiques : « The algorithm requested by this problem is one given by a real number machine in the sense of BCSS (see also Problem 3). » ; « Time is measured by the number of arithmetic operations. »

Deux écarts de forme séparent cette question de l'énoncé du théorème 1, et il faut les voir pour ne pas confondre les deux. Le système s'écrit `Ax ≥ b` chez Smale et `Ax ≤ b` dans le théorème 1 : l'un se ramène à l'autre en écrivant `(-A) x ≤ -b`. Et Smale demande une **décision** de réalisabilité, là où le théorème 1 rend un certificat pour l'**optimisation** : il le dit lui-même, « This is a decision version of the optimization problem of linear programming. »

La version fortement polynomiale est formulée un peu plus bas, et sa définition demande les deux tailles du tableau à la fois : « Closer to the main problem above over R is a similar problem asking for a “strongly polynomial algorithm” for solving these linear programming problems. This is an algorithm over Q which is polynomial time in the sense of BCSS and moreover is polynomial time in the Turing sense. » Être polynomial au sens de BCSS, c'est l'être en nombre d'opérations ; l'être au sens de Turing, c'est l'être en bits. Le théorème 1 garantit les deux ensemble.

Smale ajoute un jugement, que ce carnet ne fait que rapporter : « It is my belief that this problem number nine is the main unsolved problem of linear programming theory. »

Ce carnet n'a pas trouvé, dans le texte extrait du manuscrit, de borne numérique explicite de la forme `(m + n)^k`. L'énoncé du théorème principal est donc qualitatif ; les bornes internes vivent dans les théorèmes qui le constituent. C'est une limite de cette accrétion, et elle est notée comme telle.

## 2. Le socle : une instance de référence et un simplexe exact

Pour compter des pivots, il faut un simplexe dont on contrôle la règle. Le socle de cette série en contient un **usage**, pas un moteur : `Search/Part1-Foundations/Search-09-LinearProgramming.ipynb` résout des programmes linéaires avec PuLP, c'est-à-dire avec un solveur boîte noire, et n'affiche aucun compteur d'itérations. Ce carnet ne peut donc pas *réemployer le simplexe de Search-09* : il réemploie **son instance de production** (le programme ci-dessous, avec son optimum) et garde PuLP comme **témoin extérieur** — deux moteurs indépendants doivent rendre le même optimum.

Le moteur écrit ici est un simplexe dense en arithmétique rationnelle exacte, pour la forme `max c·x` sous `A x ≤ b`, `x ≥ 0`, avec `b ≥ 0` (la base de départ est le paquet d'écarts). Deux règles de pivot sont disponibles : **Dantzig** (plus grand coût réduit, l'heuristique historique) et **Bland** (plus petit indice de variable entrante, la règle anti-cyclage).

In [1]:
from fractions import Fraction as F
from itertools import combinations, product
from math import isqrt
import math

import pulp


def fmt(values) -> str:
    """Ecriture courte d'un vecteur de rationnels."""
    return "(" + ", ".join(str(v) for v in values) + ")"


def solve_lp(c: list, A: list, b: list, rule: str = "dantzig", max_iter: int = 200000) -> dict:
    """Simplexe exact en dense : max c.x sous A x <= b, x >= 0 (b >= 0 requis).

    rule = "dantzig" (plus grand cout reduit) ou "bland" (plus petit indice).
    Renvoie le statut, le nombre de pivots et, pour un optimum : x, valeur,
    ecarts s des contraintes, multiplicateurs duaux y.
    """
    m, d = len(A), len(c)
    n = d + m
    T = [[F(a) for a in A[i]] + [F(1) if j == i else F(0) for j in range(m)] + [F(b[i])]
         for i in range(m)]
    z = [F(x) for x in c] + [F(0)] * m + [F(0)]
    basis = [d + i for i in range(m)]
    pivots = 0
    while True:
        pos = [j for j in range(n) if z[j] > 0]
        if not pos:
            x = [F(0)] * n
            for i, bj in enumerate(basis):
                x[bj] = T[i][-1]
            return {"status": "optimal", "pivots": pivots, "x": x[:d], "value": -z[-1],
                    "y": [-z[d + i] for i in range(m)],
                    "s": [F(b[i]) - sum(F(A[i][t]) * x[t] for t in range(d)) for i in range(m)]}
        j = max(pos, key=lambda t: (z[t], -t)) if rule == "dantzig" else min(pos)
        best = None
        for i in range(m):
            if T[i][j] > 0:
                rr = T[i][-1] / T[i][j]
                if best is None or rr < best[0] or (rr == best[0] and basis[i] < basis[best[1]]):
                    best = (rr, i)
        if best is None:
            x = [F(0)] * n
            for i, bj in enumerate(basis):
                x[bj] = T[i][-1]
            return {"status": "unbounded", "pivots": pivots, "ray": j, "x": x[:d]}
        i = best[1]
        pivots += 1
        p = T[i][j]
        T[i] = [v / p for v in T[i]]
        for k in range(m):
            if k != i and T[k][j]:
                f = T[k][j]
                T[k] = [T[k][t] - f * T[i][t] for t in range(n + 1)]
        if z[j]:
            f = z[j]
            z = [z[t] - f * T[i][t] for t in range(n + 1)]
        basis[i] = j
        if pivots > max_iter:
            return {"status": "guard", "pivots": pivots}


# L'instance de production du socle Search-09 : max 3 x1 + 2 x2
c9, A9, b9 = [3, 2], [[1, 2], [2, 1]], [4, 5]
r9 = solve_lp(c9, A9, b9)
print("simplexe exact :", r9["status"], "| pivots =", r9["pivots"],
      "| x =", fmt(r9["x"]), "| valeur =", r9["value"])

prob = pulp.LpProblem("production", pulp.LpMaximize)
x1 = pulp.LpVariable("x1", lowBound=0)
x2 = pulp.LpVariable("x2", lowBound=0)
prob += 3 * x1 + 2 * x2
prob += x1 + 2 * x2 <= 4
prob += 2 * x1 + x2 <= 5
statut = prob.solve(pulp.PULP_CBC_CMD(msg=False))
print("témoin PuLP/CBC :", pulp.LpStatus[statut], "| x =", (x1.value(), x2.value()),
      "| valeur =", pulp.value(prob.objective))

# Un programme non borné, pour voir la deuxieme sortie du theoreme
rn = solve_lp([1, 0], [[0, 1]], [1])
print("non borné :", rn["status"], "| pivots =", rn["pivots"], "| point de base =", fmt(rn["x"]))

simplexe exact : optimal | pivots = 2 | x = (2, 1) | valeur = 8
témoin PuLP/CBC : Optimal | x = (2.0, 1.0) | valeur = 8.0
non borné : unbounded | pivots = 0 | point de base = (0, 0)


Les deux moteurs tombent d'accord : `x = (2, 1)`, valeur `8` — la même instance que le socle, retrouvée par un simplexe dont on peut compter les pas. Deux pivots suffisent ici.

La question de ce carnet commence exactement là : ce nombre de pivots, peut-il exploser quand les dimensions augmentent ? La section 6 montrera que oui, et de façon exponentielle. Mais avant cela, il faut savoir ce qu'un algorithme doit **rendre** — les trois certificats du théorème 1 — parce que c'est cette exigence, et non la seule valeur de l'optimum, qui rend le problème mesurable.

## 3. Première sortie : un optimum, et la complémentarité stricte

Quand l'optimum est fini, le théorème promet un couple primal–dual `x, y` **strictement complémentaire** : pour chaque contrainte `i`, l'écart `s_i = b_i − (A x)_i` et le multiplicateur `y_i` ne peuvent pas être nuls tous les deux. En pratique, un simplexe s'arrête sur un couple qui satisfait seulement la complémentarité **faible** (`s^T y = 0`, et de même `x^T z = 0` pour les coûts réduits `z = A^T y − c`). Sur un problème dégénéré, un écart nul peut cohabiter avec un multiplicateur nul.

Les deux cellules de code qui suivent mesurent les deux situations. La première prend un programme non dégénéré : chaque contrainte est soit saturée avec un multiplicateur positif, soit lâche avec un multiplicateur nul.

In [2]:
def couts_reduits(A: list, c: list, y: list) -> list:
    """Couts reduits z = A^T y - c."""
    m, d = len(A), len(c)
    return [sum(F(A[i][j]) * F(y[i]) for i in range(m)) - F(c[j]) for j in range(d)]


def rapport(A: list, b: list, c: list, x: list, s: list, y: list, etiquette: str) -> None:
    """Verifie les identites de dualite, la complementarite faible et la stricte."""
    m = len(A)
    z = couts_reduits(A, c, y)
    bdy = sum(F(b[i]) * F(y[i]) for i in range(m))
    print(etiquette)
    print("  x =", fmt(x), "| s =", fmt(s), "| y =", fmt(y))
    print("  couts réduits z =", fmt(z), "| b.y =", bdy, "| c.x =",
          sum(F(c[j]) * F(x[j]) for j in range(len(c))))
    print("  s.y =", sum(si * yi for si, yi in zip(s, y)),
          "| x.z =", sum(xj * zj for xj, zj in zip(x, z)),
          "| s + y > 0 ?", all(si + yi > 0 for si, yi in zip(s, y)),
          "| x + z > 0 ?", all(xj + zj > 0 for xj, zj in zip(x, z)))


A1, b1, c1 = [[1, 0], [0, 1]], [1, 1], [1, 1]        # max x1 + x2 sous x <= 1
r1 = solve_lp(c1, A1, b1)
print("non dégénéré :", r1["status"], "| pivots =", r1["pivots"], "| x =", fmt(r1["x"]),
      "| valeur =", r1["value"])
rapport(A1, b1, c1, r1["x"], r1["s"], r1["y"], "couple rendu par le simplexe")

non dégénéré : optimal | pivots = 2 | x = (1, 1) | valeur = 2
couple rendu par le simplexe
  x = (1, 1) | s = (0, 0) | y = (1, 1)
  couts réduits z = (0, 0) | b.y = 2 | c.x = 2
  s.y = 0 | x.z = 0 | s + y > 0 ? True | x + z > 0 ? True


Sur le programme non dégénéré, tout coïncide : les deux contraintes sont saturées (`s = (0, 0)`), les deux multiplicateurs valent `1`, la valeur primale et la valeur duale sont égales (`b.y = 2 = c.x`), et la complémentarité stricte tient (`s + y > 0` et `x + z > 0`). Il n'y a ici rien à distinguer — c'est le cas où un simplexe ordinaire satisfait déjà l'énoncé fort.

### Un programme dégénéré

Resserrons le programme : trois contraintes pour deux variables, dont deux qui se coupent au même sommet. L'optimum est dégénéré — plus de `d` contraintes y sont saturées — et c'est là que la complémentarité faible cesse d'entraîner la stricte. La cellule calcule aussi la **face duale optimale** (l'ensemble de tous les multiplicateurs optimaux), dont l'intérieur relatif fournit, lui, un couple strictement complémentaire.

In [3]:
def solve_unique(M: list, rhs: list):
    """Resout M x = rhs exactement. Renvoie None si le systeme est inconsistant
    ou n'a pas une solution unique."""
    M = [[F(v) for v in row] for row in M]
    rhs = [F(v) for v in rhs]
    nrow, ncol = len(M), len(M[0]) if M else 0
    rows = [M[i] + [rhs[i]] for i in range(nrow)]
    pivot_of, r = [], 0
    for col in range(ncol):
        p = next((i for i in range(r, nrow) if rows[i][col] != 0), None)
        if p is None:
            continue
        rows[r], rows[p] = rows[p], rows[r]
        pv = rows[r][col]
        rows[r] = [v / pv for v in rows[r]]
        for i in range(nrow):
            if i != r and rows[i][col] != 0:
                f = rows[i][col]
                rows[i] = [rows[i][t] - f * rows[r][t] for t in range(ncol + 1)]
        pivot_of.append(col)
        r += 1
        if r == nrow:
            break
    for i in range(r, nrow):
        if all(rows[i][t] == 0 for t in range(ncol)) and rows[i][ncol] != 0:
            return None
    if len(pivot_of) < ncol:
        return None
    x = [F(0)] * ncol
    for i, col in enumerate(pivot_of):
        x[col] = rows[i][ncol]
    return x


def matrix_rank(M: list) -> int:
    """Rang exact d'une matrice de rationnels, par elimination de Gauss."""
    M = [[F(v) for v in row] for row in M]
    if not M:
        return 0
    nrow, ncol = len(M), len(M[0])
    r = 0
    for col in range(ncol):
        p = next((i for i in range(r, nrow) if M[i][col] != 0), None)
        if p is None:
            continue
        M[r], M[p] = M[p], M[r]
        pv = M[r][col]
        M[r] = [v / pv for v in M[r]]
        for i in range(nrow):
            if i != r and M[i][col] != 0:
                f = M[i][col]
                M[i] = [M[i][t] - f * M[r][t] for t in range(ncol)]
        r += 1
    return r


def sommets_face_duale(A: list, b: list, c: list, valeur) -> list:
    """Sommets de la face duale optimale {y >= 0, A^T y >= c, b.y = valeur}."""
    m, d = len(A), len(A[0])
    cand = [([F(1)] * m, F(valeur))]                      # b.y = valeur (ici b = 1)
    for i in range(m):                                    # y_i = 0
        row = [F(0)] * m
        row[i] = F(1)
        cand.append((row, F(0)))
    for j in range(d):                                    # (A^T y)_j = c_j
        cand.append(([F(A[i][j]) for i in range(m)], F(c[j])))
    verts, vus = [], set()
    for combo in combinations(range(len(cand)), m):
        y = solve_unique([cand[k][0] for k in combo], [cand[k][1] for k in combo])
        if y is None or any(v < 0 for v in y):
            continue
        if any(sum(F(A[i][j]) * y[i] for i in range(m)) < F(c[j]) for j in range(d)):
            continue
        if sum(F(b[i]) * y[i] for i in range(m)) != F(valeur):
            continue
        if tuple(y) in vus:
            continue
        vus.add(tuple(y))
        verts.append(y)
    return verts


A2, b2, c2 = [[1, 0], [1, 1], [1, -1]], [1, 1, 1], [1, 0]   # max x1, optimum degenere
r2 = solve_lp(c2, A2, b2)
print("dégénéré :", r2["status"], "| pivots =", r2["pivots"], "| x =", fmt(r2["x"]),
      "| valeur =", r2["value"])
rapport(A2, b2, c2, r2["x"], r2["s"], r2["y"], "couple rendu par le simplexe")

verts = sommets_face_duale(A2, b2, c2, r2["value"])
print("sommets de la face duale optimale :", len(verts))
for v in verts:
    print("  y =", fmt(v))
ybar = [sum(v[i] for v in verts) / len(verts) for i in range(len(A2))]
rapport(A2, b2, c2, r2["x"], r2["s"], ybar, "intérieur relatif de la face duale")

dégénéré : optimal | pivots = 1 | x = (1, 0) | valeur = 1
couple rendu par le simplexe
  x = (1, 0) | s = (0, 0, 0) | y = (1, 0, 0)
  couts réduits z = (0, 0) | b.y = 1 | c.x = 1
  s.y = 0 | x.z = 0 | s + y > 0 ? False | x + z > 0 ? False
sommets de la face duale optimale : 3
  y = (0, 1, 0)
  y = (0, 1/2, 1/2)
  y = (1, 0, 0)
intérieur relatif de la face duale
  x = (1, 0) | s = (0, 0, 0) | y = (1/3, 1/2, 1/6)
  couts réduits z = (0, 1/3) | b.y = 1 | c.x = 1
  s.y = 0 | x.z = 0 | s + y > 0 ? True | x + z > 0 ? True


Le programme dégénéré montre exactement le piège. Le couple rendu par le simplexe satisfait la complémentarité faible (`s.y = 0`, `x.z = 0`) mais **pas** la stricte : les trois écarts sont nuls alors qu'un seul multiplicateur l'est, et de même `x + z > 0` échoue. La sortie du théorème ne demande pas ce couple-là : elle demande qu'il en **existe** un. Et il en existe un : la face duale optimale a `3` sommets, et leur moyenne `y = (1/3, 1/2, 1/6)` — un point de son intérieur relatif — rend `s + y > 0` et `x + z > 0` vrais, sans changer la valeur (`b.y = 1 = c.x`).

C'est une distinction de vocabulaire qui décide de la solidité d'une phrase : *un simplexe produit un couple strictement complémentaire* est faux ; *un couple strictement complémentaire existe, et l'intérieur relatif de la face duale optimale en fournit un* est ce que la mesure montre.

## 4. Deuxième sortie : un rayon qui ne se referme pas

Si l'objectif n'est pas borné, le théorème promet un point réalisable `x` et une direction `r` telle que `A r ≤ 0` (rester réalisable en avançant) et `c^T r > 0` (l'objectif croît). C'est la forme vérifiable de la non-bornitude : il suffit de contrôler deux produits matriciels, puis de vérifier qu'avancer le long de `r` ne sort jamais du domaine.

In [4]:
A3, b3, c3 = [[1, -1], [-1, 1]], [1, 1], [1, 1]   # max x1 + x2 sous |x1 - x2| <= 1
r3 = solve_lp(c3, A3, b3)
w = [F(1), F(1)]
print("simplexe :", r3["status"], "| pivots =", r3["pivots"], "| point de base =", fmt(r3["x"]))
print("rayon candidat r =", fmt(w),
      "| A r <= 0 ?", all(sum(F(A3[i][j]) * w[j] for j in range(2)) <= 0 for i in range(2)),
      "| c.r =", sum(F(c3[j]) * w[j] for j in range(2)))

for t in (10, 1000):
    x_t = [t * w[j] for j in range(2)]
    print("t =", t, "| x + t r =", fmt(x_t),
          "| réalisable ?", all(sum(F(A3[i][j]) * x_t[j] for j in range(2)) <= F(b3[i])
                               for i in range(2)),
          "| objectif =", sum(F(c3[j]) * x_t[j] for j in range(2)))

simplexe : unbounded | pivots = 1 | point de base = (1, 0)
rayon candidat r = (1, 1) | A r <= 0 ? True | c.r = 2
t = 10 | x + t r = (10, 10) | réalisable ? True | objectif = 20
t = 1000 | x + t r = (1000, 1000) | réalisable ? True | objectif = 2000


Le simplexe détecte la non-bornitude après `1` pivot, depuis le point de base `(1, 0)`. Le rayon est vérifié en deux produits : `A r ≤ 0` et `c·r = 2 > 0`. La cellule montre aussi que la direction *reste* réalisable : `x + t r` l'est pour `t = 10` comme pour `t = 1000`, avec un objectif qui croît de `20` à `2000`. La non-bornitude n'est donc pas un diagnostic, c'est un objet — n'importe qui peut la contrôler sans refaire le calcul de l'algorithme.

Une différence de convention à garder pour la section 13 : le manuscrit normalise le rayon par `c^T r > 0`, la formalisation Lean par `c·v = 1`. Les deux disent la même chose (une direction le long de laquelle l'objectif croît) ; la seconde fixe l'échelle.

## 5. Troisième sortie : une preuve d'infaisabilité, et sa taille

L'alternative de Farkas dit qu'un système `A x ≤ b` est soit réalisable, soit contredit par un vecteur `y ≥ 0` tel que `A^T y = 0` et `b^T y < 0`. L'idée : si une combinaison positive des contraintes annule toutes les variables, il reste `0 ≤ b^T y`, que `b^T y < 0` rend absurde. Le théorème du manuscrit ajoute la borne qui rend la preuve courte : `|supp y| ≤ 1 + rang A` — au plus `1 + rang` multiplicateurs non nuls, quel que soit le nombre de contraintes.

Pour la vérifier **exhaustivement**, il faut décider l'infaisabilité sans solveur : l'élimination de Fourier–Motzkin le fait en arithmétique exacte. Sur de petits systèmes à coefficients dans `{-1, 0, 1}`, on peut alors énumérer **tous** les systèmes, ne garder que les infaisables, et demander à chacun un certificat — en cherchant d'abord les supports les plus petits.

In [5]:
def fm_realisable(A: list, b: list) -> bool:
    """Realisabilite de A x <= b par elimination de Fourier-Motzkin, en exact."""
    rows = [[F(v) for v in row] + [F(bi)] for row, bi in zip(A, b)]
    ncol = len(A[0]) if A else 0
    for _ in range(ncol):
        pos = [r for r in rows if r[0] > 0]
        neg = [r for r in rows if r[0] < 0]
        new = [r[1:] for r in rows if r[0] == 0]
        for r1 in pos:
            for r2 in neg:
                f1, f2 = -r2[0], r1[0]
                new.append([f1 * r1[t] + f2 * r2[t] for t in range(1, len(r1))])
        rows = new
        if any(all(v == 0 for v in r[:-1]) and r[-1] < 0 for r in rows):
            return False
    return all(r[-1] >= 0 for r in rows if all(v == 0 for v in r[:-1]))


def certificat_farkas(A: list, b: list):
    """Cherche y >= 0 de support <= 1 + rang A avec A^T y = 0 et b.y < 0.
    Renvoie (support, valeurs, b.y, rang) ou None."""
    m, d = len(A), len(A[0])
    rang = matrix_rank(A)
    for k in range(1, min(m, rang + 1) + 1):
        for S in combinations(range(m), k):
            M = [[A[i][j] for i in S] for j in range(d)] + [[F(1)] * k]
            y = solve_unique(M, [F(0)] * d + [F(1)])
            if y is None or any(v < 0 for v in y):
                continue
            produit = sum(F(b[i]) * v for i, v in zip(S, y))
            if produit < 0:
                return S, y, produit, rang
    return None


def famille(d: int, m: int, entrees: tuple, membres: tuple) -> tuple:
    """Enumere tous les systemes a coefficients dans `entrees` et seconds membres
    dans `membres` ; compte les infaisables et verifie leur certificat."""
    total = infaisables = manquants = 0
    support_max, exces_max = 0, -1
    for coeffs in product(entrees, repeat=m * d):
        A = [list(coeffs[i * d:(i + 1) * d]) for i in range(m)]
        for b in product(membres, repeat=m):
            total += 1
            if fm_realisable(A, list(b)):
                continue
            infaisables += 1
            cert = certificat_farkas(A, list(b))
            if cert is None:
                manquants += 1
                continue
            support_max = max(support_max, len(cert[0]))
            exces_max = max(exces_max, len(cert[0]) - cert[3])
    return total, infaisables, manquants, support_max, exces_max


for d, m in ((1, 3), (2, 3)):
    tot, inf, man, smax, emax = famille(d, m, (-1, 0, 1), (-1, 0, 1))
    print(f"d = {d}, m = {m} : {tot} systèmes, {inf} infaisables, "
          f"certificats manquants = {man} | support max = {smax} | "
          f"max(support - rang) = {emax}")

cert = certificat_farkas([[1], [-1]], [0, -1])
print("instance x <= 0, -x <= -1 : support", cert[0], "| y =", fmt(cert[1]),
      "| b.y =", cert[2], "| rang A =", cert[3])

d = 1, m = 3 : 729 systèmes, 331 infaisables, certificats manquants = 0 | support max = 2 | max(support - rang) = 1


d = 2, m = 3 : 19683 systèmes, 4363 infaisables, certificats manquants = 0 | support max = 3 | max(support - rang) = 1
instance x <= 0, -x <= -1 : support (0, 1) | y = (1/2, 1/2) | b.y = -1/2 | rang A = 1


Sur les deux familles, **aucun** certificat ne manque : `0` sur `331` systèmes infaisables à une variable, `0` sur `4363` à deux variables. La borne de support tient, et elle est **atteinte** : le support maximal vaut `2` puis `3`, soit `1 + rang` lorsque la matrice est de rang plein (`1` et `2`), et l'excès **maximal** `support − rang` vaut `1` dans les deux familles — la borne n'est jamais dépassée, et son écart maximal est le même dans l'une et l'autre. L'instance `x ≤ 0, −x ≤ −1` est le cas minimal : deux contraintes contradictoires sur une variable de rang `1`, qui demandent deux multiplicateurs non nuls (`y = (1/2, 1/2)`) — aucun des deux ne suffit seul, et `b·y = −1/2 < 0`.

Deux réserves, mesurées plutôt que supposées. L'énumération porte sur des coefficients dans `{-1, 0, 1}`, donc sur une famille finie et non sur tous les systèmes rationnels. Et la borne est vérifiée **exhaustivement sur cette famille** : c'est un contrôle, pas une preuve de la borne générale — c'est le rôle de l'énoncé formalisé, lu en section 13.

Remarque utile pour la suite : le théorème dit `|supp y| ≤ 1 + rang A`. Quand `A` est de rang plein (le cas de tous les systèmes de rang `1` et `2` ci-dessus), cela signifie qu'**une preuve d'infaisabilité n'a jamais besoin de plus de `d + 1` contraintes**, quel que soit `m`. Une contrainte sur deux, sur mille, peut suffire.

## Exercice 1 — Construire un certificat de Farkas

Le programme `x ≤ 0`, `−x ≤ −2` est infaisable : la première contrainte demande `x` négatif ou nul, la seconde `x ≥ 2`.

**Objectif** : exhiber un vecteur `y ≥ 0` qui prouve l'infaisabilité, puis le contrôler avec les fonctions de la section.

**Indices**

- `# Etape 1 :` écrire `A` et `b` pour deux contraintes sur une seule variable.
- `# Etape 2 :` la combinaison `y_1 (x) + y_2 (−x)` annule la variable dès que `y_1 = y_2` ; il reste à choisir la valeur qui rend le membre de droite `y_1 · 0 + y_2 · (−2)` strictement négatif.
- `# Etape 3 :` vérifier `A^T y = 0`, `b·y < 0`, et comparer le support à `1 + rang A`.

In [6]:
A_ex1 = [[1], [-1]]
b_ex1 = [0, -2]

y_ex1 = None  # TODO etudiant : un couple (y1, y2) de rationnels non negatifs

if y_ex1 is None:
    print("Exercice a completer : le certificat n'est pas encore construit.")
else:
    produit = sum(F(A_ex1[i][0]) * F(y_ex1[i]) for i in range(2))
    droite = sum(F(b_ex1[i]) * F(y_ex1[i]) for i in range(2))
    support = sum(1 for v in y_ex1 if v != 0)
    print("y =", fmt([F(v) for v in y_ex1]),
          "| A^T y =", produit,
          "| b.y =", droite,
          "| b.y < 0 ?", droite < 0,
          "| support =", support,
          "| 1 + rang A =", 1 + matrix_rank(A_ex1),
          "| support <= 1 + rang ?", support <= 1 + matrix_rank(A_ex1))

Exercice a completer : le certificat n'est pas encore construit.


## 6. L'explosion du simplexe : Klee–Minty

Le simplexe n'est pas fortement polynomial, et il n'est même pas polynomial en les dimensions : Klee et Minty ont construit en 1972 une famille de cubes déformés où la règle de Dantzig traverse un nombre de sommets exponentiel en la dimension. Deux des textes cités par ce carnet le rappellent d'une phrase. Le manuscrit de 2026 : « Klee and Minty exhibited exponentially long runs for a classical choice of pivot ». La liste de Smale : « Klee and Minty showed that it was exponentially slow in the worst case. » Le carnet en exécute une variante mesurable.

Le cube déformé de dimension `d` est décrit par `d` inégalités `x_i + ε x_{i−1} ≤ 1` avec `0 < ε < 1`, plus `x ≥ 0`. Il a `2d` facettes et `2^d` sommets, et il est **simple** — chaque sommet est l'intersection d'exactement `d` facettes, ce qui rend le graphe des arêtes facile à reconstruire : deux sommets sont adjacents quand ils partagent `d − 1` facettes saturées.

Trois mesures distinctes suivent :

1. la **structure** du polytope (nombre de sommets, simplicité, degrés, diamètre), par énumération exacte de tous les sommets ;
2. le nombre de **pivots de la règle de Dantzig** sur une famille à coefficients cumulés, et la distance géométrique entre le départ et l'optimum ;
3. le nombre de **pivots de la règle de Bland** sur la même famille — la règle anti-cyclage, qui n'est pas la même heuristique.

In [7]:
def sommets(facettes: list, d: int) -> dict:
    """Enumere les sommets d'un polytope {a.x <= bi} : {point : facettes saturees}.
    Un sommet est l'intersection de d facettes qui le saturent toutes."""
    out = {}
    for S in combinations(range(len(facettes)), d):
        x = solve_unique([facettes[i][0] for i in S], [facettes[i][1] for i in S])
        if x is None:
            continue
        if all(sum(ai * xi for ai, xi in zip(a, x)) <= bi for a, bi in facettes):
            out[tuple(x)] = frozenset(
                i for i, (a, bi) in enumerate(facettes)
                if sum(ai * xi for ai, xi in zip(a, x)) == bi)
    return out


def graphe(pts: dict, d: int) -> dict:
    """Graphe des aretes d'un polytope simple : partager d - 1 facettes saturees."""
    clefs = list(pts)
    adj = {v: set() for v in clefs}
    for v, w in combinations(clefs, 2):
        if len(pts[v] & pts[w]) == d - 1:
            adj[v].add(w)
            adj[w].add(v)
    return adj


def distances(adj: dict, origine) -> dict:
    """Distances en nombre d'aretes depuis un sommet, par parcours en largeur."""
    dist = {origine: 0}
    front = [origine]
    while front:
        suite = []
        for u in front:
            for v in adj[u]:
                if v not in dist:
                    dist[v] = dist[u] + 1
                    suite.append(v)
        front = suite
    return dist


def diametre(adj: dict) -> int:
    """Diametre du graphe : deux parcours en largeur (exact sur ces familles)."""
    premier = next(iter(adj))
    loin = max(distances(adj, premier), key=lambda v: distances(adj, premier)[v])
    return max(distances(adj, loin).values())


def plus_long_chemin_croissant(adj: dict, valeur: dict) -> int:
    """Longueur maximale d'une suite de sommets voisins a valeur strictement croissante."""
    best = {}
    for v in sorted(adj, key=lambda t: (valeur[t], t)):
        best[v] = 1 + max([best[u] for u in adj[v] if valeur[u] < valeur[v]] or [0])
    return max(best.values())


def facettes_km(d: int, eps) -> list:
    """Cube de Klee-Minty : x_i + eps x_{i-1} <= 1 et x_i >= 0."""
    out = []
    for i in range(d):
        a = [F(0)] * d
        a[i] = F(1)
        if i > 0:
            a[i - 1] = F(eps)
        out.append((a, F(1)))
    for i in range(d):
        a = [F(0)] * d
        a[i] = F(-1)
        out.append((a, F(0)))
    return out


eps = F(1, 3)
print("cube de Klee–Minty, eps =", eps)
for d in range(1, 9):
    pts = sommets(facettes_km(d, eps), d)
    adj = graphe(pts, d)
    simple = all(len(t) == d for t in pts.values())
    degres = (min(len(s) for s in adj.values()), max(len(s) for s in adj.values()))
    print(f"d = {d} : sommets = {len(pts)} (2^d = {2 ** d}) | simple = {simple} | "
          f"degrés = {degres} | diamètre = {diametre(adj)} | n - d = {2 * d - d}")

cube de Klee–Minty, eps = 1/3
d = 1 : sommets = 2 (2^d = 2) | simple = True | degrés = (1, 1) | diamètre = 1 | n - d = 1
d = 2 : sommets = 4 (2^d = 4) | simple = True | degrés = (2, 2) | diamètre = 2 | n - d = 2
d = 3 : sommets = 8 (2^d = 8) | simple = True | degrés = (3, 3) | diamètre = 3 | n - d = 3
d = 4 : sommets = 16 (2^d = 16) | simple = True | degrés = (4, 4) | diamètre = 4 | n - d = 4
d = 5 : sommets = 32 (2^d = 32) | simple = True | degrés = (5, 5) | diamètre = 5 | n - d = 5


d = 6 : sommets = 64 (2^d = 64) | simple = True | degrés = (6, 6) | diamètre = 6 | n - d = 6


d = 7 : sommets = 128 (2^d = 128) | simple = True | degrés = (7, 7) | diamètre = 7 | n - d = 7


d = 8 : sommets = 256 (2^d = 256) | simple = True | degrés = (8, 8) | diamètre = 8 | n - d = 8


La structure est celle qu'annonce la théorie : `2^d` sommets, polytope simple, graphe `d`-régulier, et un diamètre qui vaut exactement `n − d` — la borne de Hirsch, atteinte à toutes les dimensions mesurées, de `1` à `8`, sur `256` sommets en dimension `8`. Vérifier le diamètre ne suffit pourtant pas : ce qui explose dans Klee–Minty n'est pas la distance entre deux sommets, c'est le **chemin suivi par une règle de pivot**.

### La règle de Dantzig sur un cube à coefficients cumulés

Pour faire exploser le nombre de pivots, on remplace `ε` par de petits rapports de puissances d'une base. La famille utilisée ici, en base `10`, s'écrit pour `i = 1 … d` :

`2 Σ_{j<i} 10^{i−j} x_j + x_i ≤ 10^{2(i−1)}`, avec l'objectif `Σ_j 10^{d−j} x_j`.

La forme exacte des inégalités de la note de 1972 n'est pas reproduite : la copie archivée est un scan, dont ni l'extraction de texte ni la reconnaissance optique ne rendent les formules affichées — le texte courant se lit, les inégalités non. Ce carnet travaille donc une **variante** de la famille — même géométrie de cube déformé, coefficients cumulés — qu'il **vérifie par le calcul** au lieu de la citer. La cellule de code qui suit compte les pivots de la règle de Dantzig, puis décrit la géométrie du même polytope.


In [8]:
def cube_cumule(d: int, base: int = 10) -> tuple:
    """Cube de Klee-Minty a coefficients cumules :
    2 somme_{j<i} base^(i-j) x_j + x_i <= base^(2(i-1)),
    objectif somme_j base^(d-j) x_j."""
    A, b = [], []
    for i in range(1, d + 1):
        row = [F(0)] * d
        for j in range(1, i):
            row[j - 1] = 2 * F(base) ** (i - j)
        row[i - 1] = F(1)
        A.append(row)
        b.append(F(base) ** (2 * (i - 1)))
    c = [F(base) ** (d - j) for j in range(1, d + 1)]
    return c, A, b


print("règle de Dantzig sur le cube cumulé (base 10)")
for d in range(1, 11):
    c, A, b = cube_cumule(d)
    r = solve_lp(c, A, b, rule="dantzig")
    print(f"d = {d} : {r['pivots']} pivots ({r['status']}) | 2^d - 1 = {2 ** d - 1} "
          f"| egal ? {r['pivots'] == 2 ** d - 1}")

print()
print("géométrie du même polytope")
for d in range(1, 8):
    c, A, b = cube_cumule(d)
    fac = [(list(A[i]), b[i]) for i in range(d)]
    fac += [([F(-1) if t == i else F(0) for t in range(d)], F(0)) for i in range(d)]
    pts = sommets(fac, d)
    adj = graphe(pts, d)
    valeur = {v: sum(wi * xi for wi, xi in zip(c, v)) for v in pts}
    distinctes = len(set(valeur.values())) == len(valeur)
    origine = min(pts, key=lambda v: (sum(t * t for t in v), v))
    optimum = max(pts, key=lambda v: valeur[v])
    print(f"d = {d} : sommets = {len(pts)} | simple = {all(len(t) == d for t in pts.values())} "
          f"| valeurs distinctes = {distinctes} | diamètre = {diametre(adj)} "
          f"| distance origine -> optimum = {distances(adj, origine)[optimum]} "
          f"| plus long chemin croissant = {plus_long_chemin_croissant(adj, valeur)} "
          f"| 2^d = {2 ** d}")

règle de Dantzig sur le cube cumulé (base 10)
d = 1 : 1 pivots (optimal) | 2^d - 1 = 1 | egal ? True
d = 2 : 3 pivots (optimal) | 2^d - 1 = 3 | egal ? True
d = 3 : 7 pivots (optimal) | 2^d - 1 = 7 | egal ? True
d = 4 : 15 pivots (optimal) | 2^d - 1 = 15 | egal ? True
d = 5 : 31 pivots (optimal) | 2^d - 1 = 31 | egal ? True
d = 6 : 63 pivots (optimal) | 2^d - 1 = 63 | egal ? True
d = 7 : 127 pivots (optimal) | 2^d - 1 = 127 | egal ? True
d = 8 : 255 pivots (optimal) | 2^d - 1 = 255 | egal ? True


d = 9 : 511 pivots (optimal) | 2^d - 1 = 511 | egal ? True


d = 10 : 1023 pivots (optimal) | 2^d - 1 = 1023 | egal ? True

géométrie du même polytope
d = 1 : sommets = 2 | simple = True | valeurs distinctes = True | diamètre = 1 | distance origine -> optimum = 1 | plus long chemin croissant = 2 | 2^d = 2
d = 2 : sommets = 4 | simple = True | valeurs distinctes = True | diamètre = 2 | distance origine -> optimum = 1 | plus long chemin croissant = 4 | 2^d = 4
d = 3 : sommets = 8 | simple = True | valeurs distinctes = True | diamètre = 3 | distance origine -> optimum = 1 | plus long chemin croissant = 8 | 2^d = 8
d = 4 : sommets = 16 | simple = True | valeurs distinctes = True | diamètre = 4 | distance origine -> optimum = 1 | plus long chemin croissant = 16 | 2^d = 16
d = 5 : sommets = 32 | simple = True | valeurs distinctes = True | diamètre = 5 | distance origine -> optimum = 1 | plus long chemin croissant = 32 | 2^d = 32


d = 6 : sommets = 64 | simple = True | valeurs distinctes = True | diamètre = 6 | distance origine -> optimum = 1 | plus long chemin croissant = 64 | 2^d = 64


d = 7 : sommets = 128 | simple = True | valeurs distinctes = True | diamètre = 7 | distance origine -> optimum = 1 | plus long chemin croissant = 128 | 2^d = 128


### Ce que la règle de Dantzig paie sur ce polytope

La colonne des pivots suit exactement `2^d - 1` : `1` pivot en dimension 1, `3` en dimension 2, `7` en dimension 3, et `1023` en dimension 10. La règle du plus grand coût réduit ne fait aucune erreur de choix — elle est simplement conduite sur un chemin qui double à chaque dimension ajoutée.

La seconde boucle dit quelque chose de plus dérangeant. Sur le **même** polytope :

- `2^d` sommets, tous **simples** (exactement `d` facettes actives), tous de **valeurs distinctes** ;
- diamètre égal à `d`, autrement dit le plus court chemin entre les deux extrémités reste court ;
- **distance de l'origine à l'optimum égale à `1`** à toutes les dimensions mesurées : l'optimum est un voisin, atteignable par une seule arête ;
- plus long chemin croissant de `2`, `4`, `8`, `16`, `32`, `64`, `128` pour `d` de 1 à 7, soit `2^d`.

Un algorithme qui **saurait** où se trouve l'optimum n'aurait qu'un pas à faire. Dantzig en fait `1023`. La difficulté ne vient donc ni du diamètre du polytope, ni de l'éloignement de la solution : elle vient du fait que l'information locale — quel sommet voisin améliore l'objectif — ne dit rien de la direction globale. Un cube déformé peut cacher une pente douce derrière `d - 1` pentes qui remontent.

### Le même polytope, règle de Bland

Bland choisit toujours l'indice **le plus petit** parmi les coûts réduits positifs, ce qui garantit la terminaison sans cycler. Ce critère ne regarde pas la valeur de l'amélioration : on peut donc se demander si l'évitement du cyclage se paie en pivots sur cette famille.

In [9]:
print("règle de Bland sur le même cube cumulé")
pivots_bland, statuts = [], []
for d in range(1, 17):
    c, A, b = cube_cumule(d)
    r = solve_lp(c, A, b, rule="bland")
    pivots_bland.append(r["pivots"])
    statuts.append(r["status"])
print("toutes les exécutions sont optimales ?", all(s == "optimal" for s in statuts))
print()

# Nombres de Fibonacci : F(0) = 0, F(1) = 1
fibonacci = [0, 1]
for _ in range(len(pivots_bland) + 1):
    fibonacci.append(fibonacci[-1] + fibonacci[-2])

print("d : pivots | rapport à la dimension précédente")
print("        | a(d) = a(d-1) + a(d-2) + 1 ? | 2 F(d+1) - 1 ?")
for d, p in enumerate(pivots_bland, start=1):
    if d == 1:
        print(f"d = {d} : {p}")
    elif d == 2:
        print(f"d = {d} : {p} | rapport = {p / pivots_bland[d - 2]:.4f}")
    else:
        recurrence = p == pivots_bland[d - 2] + pivots_bland[d - 3] + 1
        ferme = 2 * fibonacci[d + 1] - 1
        print(f"d = {d} : {p} | rapport = {p / pivots_bland[d - 2]:.4f} "
              f"| {recurrence} | {p == ferme}")

phi = (1 + math.sqrt(5)) / 2
print()
print(f"nombre d'or : (1 + racine de 5) / 2 = {phi:.4f}")

règle de Bland sur le même cube cumulé


toutes les exécutions sont optimales ? True

d : pivots | rapport à la dimension précédente
        | a(d) = a(d-1) + a(d-2) + 1 ? | 2 F(d+1) - 1 ?
d = 1 : 1
d = 2 : 3 | rapport = 3.0000
d = 3 : 5 | rapport = 1.6667 | True | True
d = 4 : 9 | rapport = 1.8000 | True | True
d = 5 : 15 | rapport = 1.6667 | True | True
d = 6 : 25 | rapport = 1.6667 | True | True
d = 7 : 41 | rapport = 1.6400 | True | True
d = 8 : 67 | rapport = 1.6341 | True | True
d = 9 : 109 | rapport = 1.6269 | True | True
d = 10 : 177 | rapport = 1.6239 | True | True
d = 11 : 287 | rapport = 1.6215 | True | True
d = 12 : 465 | rapport = 1.6202 | True | True
d = 13 : 753 | rapport = 1.6194 | True | True
d = 14 : 1219 | rapport = 1.6189 | True | True
d = 15 : 1973 | rapport = 1.6185 | True | True
d = 16 : 3193 | rapport = 1.6183 | True | True

nombre d'or : (1 + racine de 5) / 2 = 1.6180


### Ce que Bland paie sur le même polytope

Les comptages de Bland forment la suite `1, 3, 5, 9, 15, 25, 41, 67, 109, 177, 287, 465, 753, 1219, 1973, 3193` de la dimension 1 à la dimension 16. Deux colonnes de la sortie méritent d'être lues ensemble :

- la récurrence `a(d) = a(d-1) + a(d-2) + 1` est vérifiée à **chaque** dimension mesurée à partir de 3 ;
- la forme close `2 F(d+1) - 1`, où `F` est la suite de Fibonacci, rend **exactement** le même nombre à chaque dimension vérifiée, de `3` à `16`.

Autrement dit, sur toute la plage observée, `a(d) + 1` est deux fois un nombre de Fibonacci. Les rapports entre dimensions consécutives le confirment : `3.0000` dès la dimension 2, `1.6667` en dimension 3, puis `1.8000`, `1.6667`, `1.6667`, `1.6400`, et une descente régulière jusqu'à `1.6183` en dimension 16 — le nombre d'or vaut `1.6180`.

Une suite qui obéit à cette récurrence croît comme `phi^d` : c'est une propriété de la récurrence, pas une observation supplémentaire. Bland est donc exponentiel sur cette famille lui aussi, avec une base plus petite que celle de Dantzig. En dimension 16, le compte est de `3193` pivots là où la loi de Dantzig, vérifiée jusqu'à la dimension 10, donnerait `2^16 - 1 = 65535`.

| `d` | Dantzig (section 6) | Bland (sortie ci-dessus) |
|---|---|---|
| 3 | 7 | 5 |
| 4 | 15 | 9 |
| 5 | 31 | 15 |
| 6 | 63 | 25 |
| 7 | 127 | 41 |
| 8 | 255 | 67 |

La colonne de Dantzig reprend le comptage de la section 6, la seconde la sortie de cette section : ce sont deux règles différentes sur le **même** polytope.

Ce que ce carnet affirme, et rien de plus : sur la famille de cubes de cette section, **les deux règles de pivot examinées sont exponentielles**, et la récurrence de Bland y est vérifiée de la dimension 3 à la dimension 16 sans démonstration au-delà. Le nombre de pas ne dépend donc pas seulement de la taille `(m, n)` du programme : il dépend de la géométrie que les coefficients dessinent. C'est précisément la dépendance que la borne de la section 1 refuse — elle compte les pas en fonction de `m` et `n` seuls, quelles que soient les valeurs des coefficients.

Reste une deuxième confusion à lever, que la section suivante traite : **la dimension n'est pas la longueur de l'entrée**. Deux variables suffisent pour écrire un programme dont la description occupe des milliers de bits ; or la borne du théorème 1 compte les pas en `m` et `n` seuls, jamais dans cette longueur.

## 7. La longueur d'entrée n'est pas la dimension

Le théorème compte les pas en fonction de `m` et `n`. Il ajoute une clause sur les **nombres** : sur une entrée rationnelle, tout nombre intermédiaire a une longueur binaire polynomiale dans la longueur binaire de l'entrée. Deux quantités différentes sont donc en jeu, et il faut les mesurer séparément.

- **La dimension** : `m` contraintes, `n` variables. C'est ce que le simplexe voit quand il compte ses pivots.
- **La longueur de l'entrée** : le nombre de bits qu'il faut pour écrire tous les coefficients — numérateurs et dénominateurs. C'est ce qu'un algorithme « polynomial » au sens de la théorie de la complexité doit respecter.

Cette section pose trois questions à l'arithmétique, sans algorithme d'optimisation :

1. Combien de bits faut-il pour écrire `2^-L` ?
2. Que devient `2^-L` dans un flottant binaire64 ?
3. À dimensions égales, que font les nombres manipulés quand `L` croît ?

In [10]:
def bits_rationnel(q: F) -> int:
    """Longueur en bits d'un rationnel : numérateur et dénominateur écrits sans
    leur premier bit, un bit de signe pour chacun, un séparateur."""
    numerateur = abs(q.numerator).bit_length() - 1 if q.numerator else 0
    return 3 + numerateur + (q.denominator.bit_length() - 1)


print("2^-L : écriture rationnelle contre flottant")
for L in (10, 52, 100, 1074, 1075, 1100):
    q = F(1, 2 ** L)
    print(f"L = {L} : bits pour 2^-L = {bits_rationnel(q)} (L + 3 ? {bits_rationnel(q) == L + 3}) "
          f"| float(2^-L) = {float(q)!r}")

L = 10000
q = F(1, 2 ** L)
print(f"L = 10000 : bits pour 2^-L = {bits_rationnel(q)} (L + 3 ? {bits_rationnel(q) == L + 3})")

print()
print("séparer 0 de 2^-L par des intervalles de rayon 2^-k")
for L in (10, 100, 1000):
    k = 0
    while True:
        haut_0 = F(0) + F(1, 2 ** k)
        bas_1 = F(1, 2 ** L) - F(1, 2 ** k)
        if haut_0 < bas_1:
            break
        k += 1
    print(f"L = {L} : plus petit k à intervalles disjoints = {k} (L + 2 ? {k == L + 2}) "
          f"| bits pour 2^-k = {bits_rationnel(F(1, 2 ** k))}")

print()
print("mêmes dimensions (m = 3, n = 2), longueur des coefficients croissante")
for L in (10, 20, 40, 80, 160):
    B = 2 ** L
    At = [[-1, 0], [0, -1], [B, 1]]
    bt = [0, 0, 1]
    r = solve_lp([1, 0], At, bt)
    bits_a = max(max(F(v).numerator.bit_length(), F(v).denominator.bit_length())
                 for row in At for v in row)
    bits_x = max(max(q.numerator.bit_length(), q.denominator.bit_length()) for q in r["x"])
    print(f"L = {L} : plus grand coefficient = {bits_a} bits | simplexe : {r['status']}, "
          f"{r['pivots']} pivot | x[0] = 2^-L ? {r['x'][0] == F(1, 2 ** L)} | bits de x = {bits_x}")

2^-L : écriture rationnelle contre flottant
L = 10 : bits pour 2^-L = 13 (L + 3 ? True) | float(2^-L) = 0.0009765625
L = 52 : bits pour 2^-L = 55 (L + 3 ? True) | float(2^-L) = 2.220446049250313e-16
L = 100 : bits pour 2^-L = 103 (L + 3 ? True) | float(2^-L) = 7.888609052210118e-31
L = 1074 : bits pour 2^-L = 1077 (L + 3 ? True) | float(2^-L) = 5e-324
L = 1075 : bits pour 2^-L = 1078 (L + 3 ? True) | float(2^-L) = 0.0
L = 1100 : bits pour 2^-L = 1103 (L + 3 ? True) | float(2^-L) = 0.0
L = 10000 : bits pour 2^-L = 10003 (L + 3 ? True)

séparer 0 de 2^-L par des intervalles de rayon 2^-k
L = 10 : plus petit k à intervalles disjoints = 12 (L + 2 ? True) | bits pour 2^-k = 15
L = 100 : plus petit k à intervalles disjoints = 102 (L + 2 ? True) | bits pour 2^-k = 105
L = 1000 : plus petit k à intervalles disjoints = 1002 (L + 2 ? True) | bits pour 2^-k = 1005

mêmes dimensions (m = 3, n = 2), longueur des coefficients croissante
L = 10 : plus grand coefficient = 11 bits | simplexe : optimal,

### Lecture des trois mesures

**Écrire `2^-L` coûte `L + 3` bits.** L'égalité est vérifiée à chaque valeur testée, jusqu'à `L = 10000`, qui demande `10003` bits. Le dénominateur `2^L` s'écrit en `L + 1` bits : toute l'information tient dans sa longueur, pas dans sa valeur. La longueur de l'entrée est donc **linéaire en `L`**, et non en `2^L`.

**Le flottant, lui, s'arrête.** En binaire64, `2^-1074` vaut encore `5e-324`, et `2^-1075` vaut `0.0`. Un calcul mené en flottants ne peut plus distinguer `2^-1100` de zéro, alors que l'entrée rationnelle de `1103` bits les distingue. Ce n'est pas un détail d'implémentation : si les **tests de signe** peuvent se tromper, l'algorithme peut croire un point faisable alors qu'il ne l'est pas. Un algorithme fortement polynomial doit donc soit travailler en exact, soit savoir **augmenter la précision** jusqu'à ce que le signe soit tranché. La section suivante fait ce second choix.

**Séparer deux points demande une précision du même ordre.** Pour que l'intervalle de rayon `2^-k` centré en `0` et celui centré en `2^-L` soient disjoints, le plus petit `k` vaut `12`, `102`, `1002` pour `L = 10, 100, 1000` : toujours `L + 2`. Aucune précision fixe ne convient à toutes les entrées.

### Mêmes dimensions, nombres plus longs

Le dernier bloc garde `(m, n) = (3, 2)` constant et fait croître `L` de `10` à `160`. Trois colonnes bougent ensemble :

| `L` | plus grand coefficient | pivots du simplexe | bits de la solution |
|---|---|---|---|
| 10 | 11 | 1 | 11 |
| 20 | 21 | 1 | 21 |
| 40 | 41 | 1 | 41 |
| 80 | 81 | 1 | 81 |
| 160 | 161 | 1 | 161 |

Le nombre de **pas** ne change pas : `1` pivot à chaque fois, et la solution vaut `x[0] = 2^-L`. Mais les **nombres** transportés grandissent : la solution s'écrit sur autant de bits que le plus grand coefficient de l'entrée.

C'est ici que les deux clauses du théorème se séparent nettement :

- un algorithme est **fortement polynomial** si son nombre de pas est borné par un polynôme en `m` et `n`, **indépendamment de `L`** — c'est ce que la colonne des pivots montre sur cet exemple, et ce que la section 6 a montré que le simplexe ne garantit pas en général ;
- il faut **en plus** que les nombres intermédiaires restent courts — un polynôme en la longueur de l'entrée — sinon chaque pas pourrait coûter un temps énorme. C'est la colonne des bits, et c'est l'objet des deux sections qui suivent : la taille des nombres que produit l'ellipsoïde, puis une borne sur la taille des coordonnées d'un sommet.

## 8. L'ellipsoïde : polynomial en la longueur de l'entrée, pas en la dimension

Le papier situe lui-même l'algorithme qui l'a précédé. À propos des premières méthodes polynomiales :

> The first polynomial-time algorithms for general linear programming measure the size of a rational instance in bits. Khachiyan's ellipsoid algorithm [88, 89] draws on the cutting-plane methods of Shor and Yudin–Nemirovskii [128, 148]. In its feasibility argument, the volume of the enclosing ellipsoid falls by a controlled factor at each cut. Determinant bounds in terms of the input length tell the algorithm when that volume is too small to contain a feasible region of the required thickness [71].

Traduction : les premiers algorithmes polynomiaux mesurent la taille d'une instance rationnelle **en bits** ; à chaque coupe, le volume de l'ellipsoïde enveloppant diminue d'un facteur contrôlé ; des bornes de déterminant, exprimées en fonction de la longueur de l'entrée, disent quand ce volume est devenu trop petit pour contenir une région faisable de l'épaisseur requise.

La méthode tient en trois gestes :

1. partir d'une boule qui contient le polytope ;
2. tant que le **centre** viole une contrainte `a·z <= b`, remplacer l'ellipsoïde par le plus petit qui contienne sa moitié `a·z' <= a·z` ;
3. s'arrêter dès que le centre ne viole plus rien : c'est un point du polytope.

Deux réserves de périmètre. Ce carnet implémente la **faisabilité** — trouver un point — et non l'optimisation. Et il en donne deux versions : une version **exacte**, où les tests de signe sont tranchés en raffinant des bornes rationnelles, qui sert à mesurer la loi de volume et la taille des nombres ; une version **flottante**, où la matrice est portée sous la forme `Q = L Lᵀ` pour rester définie positive, qui sert uniquement à **compter** des coupes sur de longues séries — elle ne certifie rien.

Trois questions, dans l'ordre : de combien le volume baisse-t-il à chaque coupe ? Combien de coupes faut-il quand la taille de l'entrée grandit ? Les nombres restent-ils courts ?

In [11]:
def bornes_racine(D: F, p: int) -> tuple:
    """Encadrement rationnel a 2^-p pres : s <= racine(D) * 2^p < s + 1."""
    num, den = D.numerator, D.denominator
    s = isqrt(num * den * (1 << (2 * p)))
    return F(s, (1 << p) * den), F(s + 1, (1 << p) * den)


def intervalle_valeur(z0: list, termes: list, a: list, p: int) -> tuple:
    """Bornes de a.z, ou le centre vaut z0 + somme des v_i / racine(D_i)."""
    val = sum(F(ai) * F(zi) for ai, zi in zip(a, z0))
    bas = haut = val
    for vec, D in termes:
        c = sum(F(ai) * vi for ai, vi in zip(a, vec))
        if c == 0:
            continue
        B, H = bornes_racine(D, p)
        if c > 0:
            bas += c / H
            haut += c / B
        else:
            bas += c / B
            haut += c / H
    return bas, haut


class Ellipsoide:
    """Ellipsoide (z - centre)^T Q^-1 (z - centre) <= 1, a centre porte par des radicaux."""

    def __init__(self, n: int, R: int, z0: list = None) -> None:
        self.n = n
        self.Q = [[F(R * R) if i == j else F(0) for j in range(n)] for i in range(n)]
        self.z0 = [F(v) for v in (z0 if z0 is not None else [0] * n)]
        self.termes = []
        self.coupes = 0

    def signe_contre(self, a: list, b, pmax: int = 400) -> int:
        """+1 : a.z > b ; -1 : a.z < b ; 0 : a.z = b, ou signe non tranche a pmax bits."""
        p = 8
        while p <= pmax:
            bas, haut = intervalle_valeur(self.z0, self.termes, a, p)
            bas -= F(b)
            haut -= F(b)
            if haut < 0:
                return -1
            if bas > 0:
                return 1
            if bas == 0 and haut == 0:
                return 0
            p *= 2
        return 0

    def coupe(self, a: list) -> None:
        n = self.n
        a = [F(v) for v in a]
        Qa = [sum(self.Q[i][k] * a[k] for k in range(n)) for i in range(n)]
        D = sum(a[k] * Qa[k] for k in range(n))
        self.termes.append(([F(-1, n + 1) * Qa[i] for i in range(n)], D))
        c1 = F(n * n, n * n - 1)
        beta = F(2, n + 1)
        Q2 = [[self.Q[i][j] - beta * Qa[i] * Qa[j] / D for j in range(n)] for i in range(n)]
        self.Q = [[c1 * Q2[i][j] for j in range(n)] for i in range(n)]
        self.coupes += 1


def determinant(M: list) -> F:
    M = [list(row) for row in M]
    det = F(1)
    n = len(M)
    for k in range(n):
        p = next((i for i in range(k, n) if M[i][k] != 0), None)
        if p is None:
            return F(0)
        if p != k:
            M[k], M[p] = M[p], M[k]
            det = -det
        det *= M[k][k]
        for i in range(k + 1, n):
            f = M[i][k] / M[k][k]
            for j in range(k, n):
                M[i][j] -= f * M[k][j]
    return det


def bits_max(Q: list) -> int:
    return max(max(q.numerator.bit_length(), q.denominator.bit_length()) for row in Q for q in row)


def rapport_volume_carre(n: int) -> F:
    """gamma(n)^2 en exact : n^(2n) / ((n-1)^(n-1) (n+1)^(n+1))."""
    return F(n ** (2 * n), (n - 1) ** (n - 1) * (n + 1) ** (n + 1))


def coupes_exactes(A: list, b: list, R: int, z0: list, K: int) -> tuple:
    """Au plus K coupes en exact : renvoie det Q_k / det Q_0 et les tailles de Q_k."""
    boule = Ellipsoide(len(A[0]), R, z0)
    d0 = determinant(boule.Q)
    rapports, tailles = [], []
    for _ in range(K):
        violee = None
        for a, bi in zip(A, b):
            if boule.signe_contre(a, bi) > 0:
                violee = a
                break
        if violee is None:
            break
        boule.coupe(violee)
        rapports.append(determinant(boule.Q) / d0)
        tailles.append(bits_max(boule.Q))
    return rapports, tailles


gamma2 = rapport_volume_carre(2)
gamma = math.sqrt(float(gamma2))
print(f"rapport de volume par coupe en dimension 2 : gamma^2 = {gamma2} | gamma = {gamma}")
print(f"1 / log2(1/gamma) = {1 / math.log2(1 / gamma)} | 2 / log2(1/gamma) = {2 / math.log2(1 / gamma)}")

A8 = [[1, 2], [2, 1], [-1, 0], [0, -1]]
b8 = [4, 5, 0, 0]
rapports, tailles = coupes_exactes(A8, b8, 20, [10, 10], 10)
print()
print(f"{len(rapports)} coupes exactes sur l'instance de production, "
      f"boule initiale de rayon 20 centrée en (10, 10)")
for k, r in enumerate(rapports, 1):
    print(f"coupe {k} : det Q_k / det Q_0 = {r} | = gamma^(2k) ? {r == gamma2 ** k} "
          f"| bits max de Q_k = {tailles[k - 1]}")

rapport de volume par coupe en dimension 2 : gamma^2 = 16/27 | gamma = 0.769800358919501
1 / log2(1/gamma) = 2.6494013932779428 | 2 / log2(1/gamma) = 5.2988027865558855

10 coupes exactes sur l'instance de production, boule initiale de rayon 20 centrée en (10, 10)
coupe 1 : det Q_k / det Q_0 = 16/27 | = gamma^(2k) ? True | bits max de Q_k = 13
coupe 2 : det Q_k / det Q_0 = 256/729 | = gamma^(2k) ? True | bits max de Q_k = 16
coupe 3 : det Q_k / det Q_0 = 4096/19683 | = gamma^(2k) ? True | bits max de Q_k = 20
coupe 4 : det Q_k / det Q_0 = 65536/531441 | = gamma^(2k) ? True | bits max de Q_k = 27
coupe 5 : det Q_k / det Q_0 = 1048576/14348907 | = gamma^(2k) ? True | bits max de Q_k = 35
coupe 6 : det Q_k / det Q_0 = 16777216/387420489 | = gamma^(2k) ? True | bits max de Q_k = 64
coupe 7 : det Q_k / det Q_0 = 268435456/10460353203 | = gamma^(2k) ? True | bits max de Q_k = 94
coupe 8 : det Q_k / det Q_0 = 4294967296/282429536481 | = gamma^(2k) ? True | bits max de Q_k = 131
coupe 9 : det 

### La loi de volume, exacte

En dimension 2, `gamma^2 = 16/27`. Les dix coupes exactes multiplient chacune le déterminant de `Q` par cette fraction : la colonne « = gamma^(2k) ? » vaut `True` dix fois. Le volume d'un ellipsoïde est proportionnel à la racine carrée du déterminant, donc **chaque coupe multiplie le volume par `gamma = 0.769800358919501`**, quelle que soit la contrainte qui a servi à couper.

Ce rapport ne dépend pas de la direction de la coupe, et l'on voit pourquoi sur la formule de mise à jour. La matrice devient

`Q' = n^2 / (n^2 - 1) * (Q - 2/(n+1) * Q a a^T Q / (a^T Q a))`.

Dans la base où `Q` est l'identité et `a` porte sur le premier axe, la parenthèse vaut `(n - 1)/(n + 1)` le long de `a` et `1` sur les `n - 1` autres axes. Le rapport des déterminants vaut donc `n^(2n) / ((n - 1)^(n - 1) (n + 1)^(n + 1))` : une fonction de la seule dimension.

Il faut `1 / log2(1/gamma) = 2.6494013932779428` coupes pour retirer un bit au logarithme binaire du volume. Le nombre de coupes est ainsi gouverné par un seul rapport : celui entre le volume de départ et le plus petit volume que peut occuper une région faisable qu'on ne veut pas manquer. Or la section 7 a mesuré à quel point une telle région devient étroite : séparer `0` de `2^-L` y demande des intervalles de rayon `2^-(L + 2)`, un rayon qui rétrécit quand `L` grandit. **Le comptage de l'ellipsoïde dépend de `L`** — c'est ce que les deux séries ci-dessous mesurent.

### La taille des nombres, elle aussi

La dernière colonne donne la longueur du plus grand numérateur ou dénominateur de `Q_k` : `13`, `16`, `20`, `27`, `35`, `64`, `94`, `131`, `275`, `457` bits. Dix coupes exactes, sans aucun arrondi, multiplient la taille des nombres par environ `35` — de `13` bits après la première coupe à `457` après la dixième.

Une implémentation exacte et naïve voit donc ses nombres s'allonger à chaque coupe. C'est la raison pour laquelle le théorème de la section 1 ne se contente pas de compter les pas : il porte une **clause explicite sur la longueur des nombres intermédiaires**.

### Deux séries, en flottant

La version exacte est trop lourde pour de longues séries ; la version flottante prend le relais pour **compter**. Deux expériences :

1. **Polytope fixé, boule de départ de plus en plus grande.** L'instance de production ne change pas ; le rayon initial `2R` et le centre de départ `(R, R)` s'éloignent. Le comptage suit-il le logarithme de `R` ?
2. **Dimension fixée, entrée de plus en plus longue.** Le triangle `2^L x_0 + x_1 <= 1`, `x >= 0`, d'aire `2^-(L+1)` : `(m, n)` constant, `L` croissant. Le comptage suit-il `L` ?

Le test de violation utilise un seuil **relatif** à l'échelle des termes sommés : un seuil absolu accepterait un point hors du polytope dès que celui-ci devient plus étroit que le seuil.

In [12]:
def ellipsoide_flottant(A: list, b: list, R: int, z0: list, tol: float = 1e-12,
                        cap: int = 100000) -> tuple:
    """Meme iteration en flottant, matrice portee sous la forme Q = L L^T.

    Sert a compter les coupes, pas a certifier : la forme L L^T evite la perte
    du caractere defini positif de la mise a jour naive apres quelques dizaines
    de coupes. Renvoie (trouve, coupes, centre).
    """
    n = len(A[0])
    L = [[math.sqrt(float(R) * R) if i == j else 0.0 for j in range(n)] for i in range(n)]
    z = [float(v) for v in z0]
    coupes = 0
    s = 1.0 - math.sqrt(1.0 - 2.0 / (n + 1))
    echelle = math.sqrt(n * n / (n * n - 1.0))
    while coupes < cap:
        k = None
        for i, (a, bi) in enumerate(zip(A, b)):
            termes = [float(ai) * zi for ai, zi in zip(a, z)]
            echelle_i = abs(float(bi)) + sum(abs(t) for t in termes)
            if sum(termes) - float(bi) > tol * echelle_i:
                k = i
                break
        if k is None:
            return True, coupes, z
        a = [float(v) for v in A[k]]
        h = [sum(L[i][j] * a[i] for i in range(n)) for j in range(n)]
        nh = math.sqrt(sum(v * v for v in h))
        hhat = [v / nh for v in h]
        z = [z[i] - sum(L[i][j] * hhat[j] for j in range(n)) / (n + 1) for i in range(n)]
        L = [[echelle * (L[i][j] - s * sum(L[i][t] * hhat[t] for t in range(n)) * hhat[j])
              for j in range(n)] for i in range(n)]
        coupes += 1
    return False, coupes, z


def pente(xs: list, ys: list) -> tuple:
    """Pente et ordonnee a l'origine de la droite des moindres carres."""
    mx = sum(xs) / len(xs)
    my = sum(ys) / len(ys)
    vx = sum((x - mx) ** 2 for x in xs)
    p = sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / vx
    return p, my - p * mx


A8f = [[F(1), F(2)], [F(2), F(1)], [F(-1), F(0)], [F(0), F(-1)]]
b8f = [F(4), F(5), F(0), F(0)]

print("polytope fixe, boule initiale de rayon 2R centrée en (R, R)")
xs, ys = [], []
for R in (10, 30, 100, 300, 1000, 10 ** 4, 10 ** 6, 10 ** 9):
    trouve, coupes, z = ellipsoide_flottant(A8f, b8f, 2 * R, [R, R])
    xs.append(math.log2(R))
    ys.append(coupes)
    print(f"R = {R} : point trouvé ? {trouve} | coupes = {coupes} | log2(R) = {math.log2(R):.2f}")
p, o = pente(xs, ys)
print(f"pente des moindres carrés = {p:.3f} coupes par bit de R (ordonnée à l'origine {o:.1f})"
      f" | 2 / log2(1/gamma) = {2 / math.log2(1 / gamma):.3f}")

print()
print("dimension fixée, triangle 2^L x0 + x1 <= 1, x >= 0, boule initiale de rayon 2")
xs, ys = [], []
for L in (10, 20, 40, 80, 160):
    BL = 2 ** L
    At = [[F(-1), F(0)], [F(0), F(-1)], [F(BL), F(1)]]
    trouve, coupes, z = ellipsoide_flottant(At, [F(0), F(0), F(1)], 2, [1, 1])
    xs.append(L)
    ys.append(coupes)
    print(f"L = {L} : point trouvé ? {trouve} | coupes = {coupes} "
          f"| aire du triangle = 1/{2 * BL}")
p, o = pente(xs, ys)
print(f"pente des moindres carrés = {p:.3f} coupes par bit de L (ordonnée à l'origine {o:.1f})"
      f" | 1 / log2(1/gamma) = {1 / math.log2(1 / gamma):.3f}")

polytope fixe, boule initiale de rayon 2R centrée en (R, R)
R = 10 : point trouvé ? True | coupes = 11 | log2(R) = 3.32
R = 30 : point trouvé ? True | coupes = 18 | log2(R) = 4.91
R = 100 : point trouvé ? True | coupes = 17 | log2(R) = 6.64
R = 300 : point trouvé ? True | coupes = 38 | log2(R) = 8.23
R = 1000 : point trouvé ? True | coupes = 38 | log2(R) = 9.97
R = 10000 : point trouvé ? True | coupes = 54 | log2(R) = 13.29
R = 1000000 : point trouvé ? True | coupes = 99 | log2(R) = 19.93
R = 1000000000 : point trouvé ? True | coupes = 141 | log2(R) = 29.90
pente des moindres carrés = 5.083 coupes par bit de R (ordonnée à l'origine -9.1) | 2 / log2(1/gamma) = 5.299

dimension fixée, triangle 2^L x0 + x1 <= 1, x >= 0, boule initiale de rayon 2
L = 10 : point trouvé ? True | coupes = 16 | aire du triangle = 1/2048
L = 20 : point trouvé ? True | coupes = 50 | aire du triangle = 1/2097152
L = 40 : point trouvé ? True | coupes = 104 | aire du triangle = 1/2199023255552
L = 80 : point trouvé

### Les deux pentes

**Boule de départ de plus en plus grande.** Le comptage vaut `11`, `18`, `17`, `38`, `38`, `54`, `99`, `141` pour `R` de `10` à `10^9`, et la pente des moindres carrés est de `5.083` coupes par bit de `R` — contre `2 / log2(1/gamma) = 5.299` prédites par la loi de volume. Le facteur 2 se lit directement : doubler `R` quadruple le volume initial en dimension 2, soit **deux** bits de logarithme de volume, à raison de `1 / log2(1/gamma)` coupes par bit.

Les valeurs ne sont **pas** monotones : `18` coupes pour `R = 30` contre `17` pour `R = 100`, et `38` deux fois de suite. L'algorithme s'arrête au moment où le centre entre dans le polytope, et ce moment dépend du chemin suivi, pas seulement du volume. C'est la **tendance** qui est linéaire en `log2(R)`, pas chaque valeur.

**Entrée de plus en plus longue.** Le comptage vaut `16`, `50`, `104`, `205`, `422` pour `L` de `10` à `160`, et la pente vaut `2.676` coupes par bit de `L` — contre `1 / log2(1/gamma) = 2.649` prédites. Chaque bit de `L` divise l'aire du triangle par deux, c'est-à-dire retire un bit au logarithme du volume : une coupe retire `log2(1/gamma) = 0.3774` bit, l'inverse de `2.649`, donc il en faut `2.649` par bit de `L`.

Les deux pentes suivent la même loi, avec le facteur 2 qui sépare une dilatation de rayon d'une dilatation de volume.

### Retour sur la section 7

Les deux algorithmes, côte à côte sur les mêmes instances — le triangle `2^L x_0 + x_1 <= 1`, `x >= 0`, `(m, n) = (3, 2)` constant. Les deux premières colonnes numériques reprennent la sortie de la section 7, la dernière la seconde série ci-dessus :

| `L` | plus grand coefficient | simplexe : pivots | ellipsoïde : coupes |
|---|---|---|---|
| 10 | 11 bits | 1 | 16 |
| 20 | 21 bits | 1 | 50 |
| 40 | 41 bits | 1 | 104 |
| 80 | 81 bits | 1 | 205 |
| 160 | 161 bits | 1 | 422 |

Sur cette famille, le simplexe ne voit pas `L` : un pivot à chaque longueur d'entrée. L'ellipsoïde, lui, le voit très bien : son comptage est linéaire en `L`.

Ce n'est pas un classement entre les deux algorithmes, c'est la mesure de deux dépendances différentes. Le simplexe dépend de la **géométrie** du polytope — la section 6 l'a montré avec un comptage en `2^d - 1` sur un cube de dimension `d`. L'ellipsoïde dépend de la **longueur des nombres** — c'est ce que cette section mesure, et c'est ce qui lui vaut sa garantie polynomiale en temps. Aucun des deux n'a, sur les exemples de ce carnet, un comptage borné par un polynôme en `(m, n)` : celui du simplexe explose en la dimension, celui de l'ellipsoïde croît à dimensions fixées. La question de Smale était précisément de savoir si ces deux nombres suffisent, et la réponse du papier est oui — par un algorithme qui n'est ni l'un ni l'autre.

## Exercice 2 — l'ellipsoïde sur le cube de Klee–Minty

La section 6 a construit un cube où Dantzig prend `2^d - 1` pivots. Que fait l'ellipsoïde sur le **même** polytope ?

L'ellipsoïde ne suit pas les arêtes : il coupe par des hyperplans, et le chemin combinatoire du simplexe ne devrait donc pas l'atteindre. Mais ce cube a une autre particularité : ses coefficients valent `10^(2(d-1))` au plus, donc sa longueur d'entrée croît avec `d`. La question de l'exercice est de savoir laquelle des deux quantités — la géométrie ou la longueur des nombres — gouverne son comptage.

**Consigne.** Pour `d` de `2` à `8` :

1. `# Etape 1` — reprendre les facettes de `cube_cumule(d)` et leur ajouter les `d` contraintes `x_i >= 0`.
2. `# Etape 2` — partir du centre de la boîte, `h = 10^(2(d-1))` et `z0 = (h/2, ..., h/2)`, avec une boule initiale de rayon `R = d * h`.
3. `# Etape 3` — compter les coupes de `ellipsoide_flottant` et les comparer à `2^d - 1`.

Le test de violation étant relatif, le point rendu peut violer une contrainte de l'ordre de `1e-12` fois son échelle : on ne certifie rien ici, on compte.


In [13]:
def facettes_cube(d: int, base: int = 10) -> tuple:
    """Cube cumule de la section 6, complete par les d facettes x_i >= 0."""
    c, A, b = cube_cumule(d, base)
    A2 = list(A)  # TODO etudiant : ajouter les d lignes -x_i <= 0
    b2 = list(b)  # TODO etudiant : ajouter les d zeros correspondants
    return A2, b2


coupes_cube = None  # TODO etudiant : {d: nombre de coupes de l'ellipsoide} pour d = 2..8

if coupes_cube is None:
    print("Exercice a completer : compter les coupes de l'ellipsoide sur le cube cumule,")
    print("puis comparer a 2^d - 1 pivots de la regle de Dantzig.")
else:
    print("d : coupes de l'ellipsoide | 2^d - 1 pivots de la regle de Dantzig")
    precedent = None
    for d in sorted(coupes_cube):
        coupes = coupes_cube[d]
        rapport = "" if precedent is None else f" | rapport = {coupes / precedent:.2f}"
        print(f"d = {d} : {coupes} | {2 ** d - 1}{rapport}")
        precedent = coupes


Exercice a completer : compter les coupes de l'ellipsoide sur le cube cumule,
puis comparer a 2^d - 1 pivots de la regle de Dantzig.


## 9. Ce que pèse un sommet

Les sections précédentes ont mesuré des **comptages**. Celle-ci mesure autre chose : la **taille des nombres**, et ce que cette taille impose à n'importe quel algorithme.

Un sommet d'un polyèdre rationnel est la solution d'un système carré dont les coefficients sont des entrées du problème. Par la règle de Cramer, chaque coordonnée est un quotient de deux déterminants extraits de la matrice des contraintes. L'inégalité de Hadamard borne chacun d'eux : un déterminant `n` par `n` dont les colonnes ont des normes au plus `sqrt(n) M` vaut au plus `(sqrt(n) M)^n`. Si les entrées tiennent sur `L` bits, cela donne en bits `n L + (n / 2) log2(n)`, c'est-à-dire `2L + 1` en dimension 2.

C'est ce dont l'ellipsoïde se sert — ses bornes de déterminants lui disent quand le volume devient trop petit pour contenir une région faisable de l'épaisseur requise. C'est aussi ce qu'un algorithme fortement polynomial ne peut pas se permettre d'utiliser : son nombre de pas doit ignorer `L`.

La mesure porte sur une famille de quadrilatères en dimension 2 à coefficients de `L` bits :

- `x >= 0` et `y >= 0` ;
- `(2^L - 2) x + y <= 1` ;
- `3 x + (2^L - 2) y <= 1`.

Chacune des deux dernières contraintes coupe un axe en `1 / (2^L - 2)`, et elles se coupent l'une l'autre en un quatrième sommet dont les coordonnées sont justement des quotients de déterminants `2` par `2`.


In [14]:
def polytope_hadamard(L: int) -> list:
    """Quadrilatere de dimension 2 a coefficients de L bits."""
    X = 2 ** L - 2
    return [([F(-1), F(0)], F(0)), ([F(0), F(-1)], F(0)),
            ([F(X), F(1)], F(1)), ([F(3), F(X)], F(1))]


def longueur(q: F) -> int:
    """Plus grande longueur, en bits, du numerateur et du denominateur."""
    return max(q.numerator.bit_length(), q.denominator.bit_length())


print("L | sommets | plus grande longueur | 2L | borne 2L + 1 | sous la borne ?")
for L in (10, 50, 100, 200):
    pts = sommets(polytope_hadamard(L), 2)
    plus_grand = max(longueur(q) for v in pts for q in v)
    print(f"L = {L} : {len(pts)} sommets | {plus_grand} bits | {2 * L} | "
          f"{2 * L + 1} | {plus_grand <= 2 * L + 1}")

X = 2 ** 10 - 2
det = X * X - 3
print()
print(f"L = 10 : determinant du systeme 2 x 2 = {det} ({det.bit_length()} bits)")
print(f"         sommet non trivial = ({X - 1} / {det}, {X - 3} / {det})")
print(f"         les trois autres sommets sont (0, 0), (0, 1 / {X}) et (1 / {X}, 0)")

L | sommets | plus grande longueur | 2L | borne 2L + 1 | sous la borne ?
L = 10 : 4 sommets | 20 bits | 20 | 21 | True
L = 50 : 4 sommets | 100 bits | 100 | 101 | True
L = 100 : 4 sommets | 200 bits | 200 | 201 | True
L = 200 : 4 sommets | 400 bits | 400 | 401 | True

L = 10 : determinant du systeme 2 x 2 = 1044481 (20 bits)
         sommet non trivial = (1021 / 1044481, 1019 / 1044481)
         les trois autres sommets sont (0, 0), (0, 1 / 1022) et (1 / 1022, 0)


### Ce que la mesure montre

`4` sommets à chaque fois, et la plus grande longueur mesurée vaut `20`, `100`, `200` puis `400` bits — exactement `2L`, sous la borne `2L + 1`. Pour `L = 10`, le sommet non trivial est `(1021 / 1044481, 1019 / 1044481)` : un dénominateur de 20 bits, égal au déterminant `1022^2 - 3` du système.

La conséquence à ne pas manquer : cette longueur n'est **pas** un défaut d'algorithme, elle est **dans la réponse**. Doubler `L` double la longueur du dénominateur du sommet, et aucun algorithme ne peut écrire une fraction dont le dénominateur occupe `2L` bits en moins de `2L` bits.

C'est la lecture exacte du théorème, et elle a deux étages qu'il ne faut pas confondre :

| Ce que le théorème borne | En quoi | Dépend de `L` ? |
|---|---|---|
| le nombre de pas | polynôme en `m` et `n` | non |
| la longueur des nombres, sur entrée rationnelle | polynôme en la longueur binaire de l'entrée | oui — `2L` sur la famille mesurée |

Confondre les deux, c'est s'attendre à ce qu'un algorithme fortement polynomial rende des réponses courtes. Il ne le peut pas quand l'entrée est longue, et il n'a pas à le faire : ce qu'il garantit, c'est que le **nombre de ses pas** ne croît pas pour autant.

L'ellipsoïde, lui, paie les deux : son comptage de coupes croît avec `L` (section 8), et ses nombres s'allongent à chaque coupe — de `13` bits après la première à `457` après la dixième, dans la série exacte de cette même section. C'est la dépendance en `L` du nombre de pas que la question de Smale demandait de dépasser.

## 10. Le déséquilibre des circuits

Jusqu'ici deux mesures ont expliqué les bornes : la **géométrie** du polytope pour le simplexe, la **longueur des nombres** pour l'ellipsoïde. Les algorithmes intermédiaires, eux, dépendaient d'une troisième : l'*état* de la matrice des contraintes. Le papier la décrit en ces termes :

> A circuit of A is a nonzero kernel vector of minimal support. The circuit imbalance is the largest ratio of magnitudes of two nonzero coordinates in the same circuit [124, 50].

Un **circuit** est donc un vecteur non nul du noyau de `A`, dont aucune coordonnée ne peut être annulée sans le rendre nul : son support est minimal. Le **déséquilibre** est le plus grand rapport entre deux coordonnées non nulles d'un même circuit. Une matrice dont tous les circuits ont des coordonnées comparables est bien conditionnée ; une matrice avec un circuit où une coordonnée vaut `u` et l'autre `1`, avec `u` minuscule, ne l'est pas.

On pourrait espérer réparer cela en changeant les unités, c'est-à-dire en multipliant chaque colonne par un facteur positif. Le papier montre que non, dès la dimension 2. La matrice qu'il exhibe est `A = [[1, 0, 1, u], [0, 1, u, 1]]`, et le passage se lit ainsi :

> Even the best rescaling can leave large ratios in a fixed dimension. For 0 < u < 1, the matrix A = [1 0 1 u; 0 1 u 1] has circuits (−1, −u, 1, 0) and (−u, −1, 0, 1). The ratios of the magnitudes of their first two coordinates are 1/u and u. Any positive diagonal column scaling multiplies these two ratios by the same factor, so at least one ratio or its reciprocal is at least 1/u.

L'argument tient en une ligne. Multiplier la colonne `j` par `d_j` divise par `d_j` la coordonnée de rang `j` de tout circuit : les deux rapports des premières coordonnées se trouvent multipliés **par le même facteur** `d2 / d1`. Si ce facteur vaut au moins `1`, le premier rapport, `1/u`, ne descend pas ; s'il est plus petit que `1`, c'est l'inverse du second, `1 / (u d2 / d1)`, qui dépasse `1/u`. Aucun choix ne rabaisse les deux à la fois.

La mesure vérifie cette affirmation sur la matrice de l'article, en `u = 3/10` : on énumère ses circuits, puis on recalcule le déséquilibre sur la matrice échelonnée.


In [15]:
def circuits(A: list) -> list:
    """Circuits de A : vecteurs du noyau de support minimal, normalises a 1 sur
    une coordonnee hors d'un sous-ensemble independant de colonnes."""
    m, d = len(A), len(A[0])
    r = matrix_rank(A)
    out = []
    for S in combinations(range(d), r + 1):
        sub = [[F(A[i][j]) for j in S] for i in range(m)]
        if matrix_rank(sub) != r:
            continue
        # minimalite : retirer une colonne quelconque fait tomber le rang
        if any(matrix_rank([[row[t] for t in range(len(S)) if t != k] for row in sub]) < r
               for k in range(len(S))):
            continue
        vec = None
        for T in combinations(range(len(S)), r):
            blk = [[sub[i][t] for t in T] for i in range(m)]
            if matrix_rank(blk) != r:
                continue
            reste = [t for t in range(len(S)) if t not in T][0]
            vT = solve_unique(blk, [-sub[i][reste] for i in range(m)])
            if vT is None:
                continue
            vec = [F(0)] * len(S)
            for pos, t in enumerate(T):
                vec[t] = vT[pos]
            vec[reste] = F(1)
            break
        if vec is None:
            continue
        plein = [F(0)] * d
        for pos, j in enumerate(S):
            plein[j] = vec[pos]
        out.append((S, plein))
    return out


def desequilibre(vec: list) -> F:
    """Rapport de la plus grande a la plus petite coordonnee non nulle."""
    nz = [abs(v) for v in vec if v != 0]
    return max(nz) / min(nz)


def desequilibre_matrice(A: list) -> F:
    """Desequilibre de circuits d'une matrice : le plus grand parmi ses circuits."""
    return max(desequilibre(v) for _, v in circuits(A))


u = F(3, 10)
A = [[F(1), F(0), F(1), u], [F(0), F(1), u, F(1)]]
print("A =", [[str(v) for v in row] for row in A], "| u =", u, "| 1/u =", 1 / u)
cs = circuits(A)
print(f"nombre de circuits = {len(cs)} | déséquilibre de A = {desequilibre_matrice(A)}")
for S, v in cs:
    residu = [sum(A[i][j] * v[j] for j in range(4)) for i in range(2)]
    print(f"  support {S} : v = {[str(x) for x in v]} | A v = {[str(x) for x in residu]}"
          f" | déséquilibre = {desequilibre(v)}")

print()
print("échelonnage diagonal : colonne j multipliée par D[j]")
print("rapport 1 et rapport 2 = rapports des deux premières coordonnées des deux circuits nommés")
for D in ([1, 1, 1, 1], [2, 1, 3, 1], [1, 7, 1, 5], [10, 1, 1, 1], [1, 10, 1, 1]):
    AD = [[A[i][j] * D[j] for j in range(4)] for i in range(2)]
    r1 = abs(F(1) / D[0]) / abs(u / D[1])
    r2 = abs(u / D[0]) / abs(F(1) / D[1])
    des = desequilibre_matrice(AD)
    print(f"  D = {D} : rapport 1 = {r1}, rapport 2 = {r2} | rapport 1 x rapport 2 = {r1 * r2}"
          f" | déséquilibre de A D recalculé = {des} | >= 1/u ? {des >= 1 / u}")

A = [['1', '0', '1', '3/10'], ['0', '1', '3/10', '1']] | u = 3/10 | 1/u = 10/3
nombre de circuits = 4 | déséquilibre de A = 10/3
  support (0, 1, 2) : v = ['-1', '-3/10', '1', '0'] | A v = ['0', '0'] | déséquilibre = 10/3
  support (0, 1, 3) : v = ['-3/10', '-1', '0', '1'] | A v = ['0', '0'] | déséquilibre = 10/3
  support (0, 2, 3) : v = ['91/30', '0', '-10/3', '1'] | A v = ['0', '0'] | déséquilibre = 10/3
  support (1, 2, 3) : v = ['0', '-91/100', '-3/10', '1'] | A v = ['0', '0'] | déséquilibre = 10/3

échelonnage diagonal : colonne j multipliée par D[j]
rapport 1 et rapport 2 = rapports des deux premières coordonnées des deux circuits nommés
  D = [1, 1, 1, 1] : rapport 1 = 10/3, rapport 2 = 3/10 | rapport 1 x rapport 2 = 1 | déséquilibre de A D recalculé = 10/3 | >= 1/u ? True
  D = [2, 1, 3, 1] : rapport 1 = 5/3, rapport 2 = 3/20 | rapport 1 x rapport 2 = 1/4 | déséquilibre de A D recalculé = 10 | >= 1/u ? True
  D = [1, 7, 1, 5] : rapport 1 = 70/3, rapport 2 = 21/10 | rapport 1 x

### Ce que la mesure montre

La matrice a `4` circuits, et chacun a un déséquilibre exactement égal à `10/3 = 1/u`. Les deux premiers sont ceux que le papier nomme, à la normalisation près : `(-1, -3/10, 1, 0)` et `(-3/10, -1, 0, 1)`, tous deux dans le noyau de `A`.

L'échelonnage ne répare rien :

| `D` | rapport 1 | rapport 2 | déséquilibre recalculé |
|---|---|---|---|
| `(1, 1, 1, 1)` | `10/3` | `3/10` | `10/3` |
| `(2, 1, 3, 1)` | `5/3` | `3/20` | `10` |
| `(1, 7, 1, 5)` | `70/3` | `21/10` | `70/3` |
| `(10, 1, 1, 1)` | `1/3` | `3/100` | `100/3` |
| `(1, 10, 1, 1)` | `100/3` | `3` | `100/3` |

La quatrième ligne est la plus parlante. Multiplier la première colonne par `10` rabaisse le premier rapport à `1/3` — son inverse, `3`, passe **sous** `1/u`. Le premier circuit est réparé. Mais le second rapport tombe à `3/100`, dont l'inverse vaut `100/3` : le déséquilibre recalculé de la matrice est `100/3`, dix fois la valeur de départ. La ligne suivante prend le facteur dans l'autre sens : le second rapport monte à `3`, et c'est le premier qui vaut `100/3` — dix fois `1/u`, sans que rien ait été rabattu.

Une réserve honnête : la borne du papier ne concerne que les deux circuits nommés. La deuxième ligne le montre — leurs rapports sont modestes (`5/3` et `3/20`), mais le déséquilibre recalculé vaut `10`, porté par un circuit que l'argument ne cite pas. La borne est valable sans être serrée : le déséquilibre réel peut être pire que ce qu'elle annonce.

### Pourquoi cela compte pour la question de Smale

Plusieurs algorithmes antérieurs tirent leur borne d'une mesure de conditionnement de la matrice ; le papier en cite deux lignées. Vavasis et Ye : un algorithme de suivi de chemin « whose iteration bound depends on the dimensions and the logarithm of a condition measure of A ». Dadush, Huiberts, Natura et Végh : une borne d'itérations « in terms of the best condition achievable by diagonal scaling of the columns ». Or aucun échelonnage de colonnes ne fait descendre le déséquilibre sous `1/u` — c'est l'argument ci-dessus — et `u` peut être aussi petit qu'on veut avec les **mêmes** dimensions `m = 2`, `n = 4`. Une borne qui en dépend n'est donc pas une fonction de `(m, n)` seule.

L'algorithme du théorème échelonne aussi, mais pour une autre raison, et le papier distingue les deux usages : dans les algorithmes antérieurs, « a change of scale that improves the geometry » ; ici, « a rescaling selects the coordinates on which a primal or dual witness can be detected » — un témoin, primal ou dual, que l'échelonnage rend visible. Sa section 2 explique « how the algorithm combines such observations, even when the initial system has arbitrarily large circuit imbalance ».


## Exercice 3 — un seul circuit, un échelonnage qui l'équilibre

La matrice de l'article résiste à tout échelonnage parce que ses deux circuits nommés partagent les deux premières colonnes. Que se passe-t-il avec une matrice qui n'a qu'un seul circuit ?

On prend

```
B = [[1, 1, u],
     [0, 1, 1]]
```

avec `u = 1/2`.

**Consigne.**

1. `# Etape 1` — énumérer les circuits de `B` avec `circuits`, et vérifier que chaque support est de taille `rang + 1`.
2. `# Etape 2` — trouver un échelonnage diagonal `D`, c'est-à-dire trois rationnels positifs `[d0, d1, d2]`, tel que le déséquilibre de circuits de `B D` vaille `1`.
3. `# Etape 3` — expliquer en une phrase pourquoi aucun `D` ne fait la même chose pour la matrice `A` de la section 10.

`# Indice` — un circuit dont toutes les coordonnées non nulles ont même valeur absolue a un déséquilibre de `1`. Par quelle colonne faut-il multiplier, et par combien ?

In [16]:
u_ex3 = F(1, 2)
B = [[F(1), F(1), u_ex3], [F(0), F(1), F(1)]]

circuits_B = None  # TODO etudiant : circuits(B)
echelle = None     # TODO etudiant : liste de trois rationnels positifs [d0, d1, d2]

if circuits_B is None or echelle is None:
    print("Exercice a completer : enumerer les circuits de B, puis trouver un echelonnage")
    print("diagonal D tel que le desequilibre de circuits de B D vaille 1.")
else:
    rang = matrix_rank(B)
    for S, v in circuits_B:
        produit = [sum(B[i][j] * v[j] for j in range(3)) for i in range(2)]
        print(f"support {S} de taille {len(S)} (rang + 1 = {rang + 1}) | v = {[str(x) for x in v]}"
              f" | B v = {[str(x) for x in produit]} | déséquilibre = {desequilibre(v)}")
    BD = [[B[i][j] * echelle[j] for j in range(3)] for i in range(2)]
    print(f"après échelonnage par D = {[str(x) for x in echelle]} :"
          f" déséquilibre = {desequilibre_matrice(BD)}")

Exercice a completer : enumerer les circuits de B, puis trouver un echelonnage
diagonal D tel que le desequilibre de circuits de B D vaille 1.


## 11. La conjecture de Hirsch, et ce qu'elle n'est pas

Le simplexe suit les arêtes. Une question naturelle est donc de savoir si le graphe d'un polytope est, en soi, court. C'est la question posée par Hirsch en 1957, et à laquelle le papier compagnon apporte une réponse négative — dans sa version polynomiale.

**Définitions.** Le diamètre d'un polytope est la longueur maximale, sur toutes les paires de sommets, d'un plus court chemin dans son graphe sommets-arêtes. La conjecture de Hirsch affirme qu'un polyèdre de dimension `d` à `n` facettes a un diamètre d'au plus `n - d`.

L'histoire tient en trois temps, tous lus dans le papier compagnon. Santos a réfuté la borne `n - d` : un polytope de dimension 43, à 86 facettes, dont deux sommets sont à au moins 44 pas l'un de l'autre. Matschke, Santos et Weibel ont ensuite ramené la dimension à 20. Mais « That example left open the possibility that some polynomial in n bounds all such distances. » C'est cette version affaiblie, la **conjecture de Hirsch polynomiale**, que le papier compagnon réfute :

> Theorem 1.1 (Superpolynomial diameter). For each positive integer-valued s(n) = n^{o(1)} there is a family of bounded simple rational polytopes P_n, one for every sufficiently large n, with exactly n facets, dimension n^{o(1)}, and s(n)-move diameter n^{ω(1)}. [...] Taking s(n) = 1 disproves the polynomial Hirsch conjecture.

Deux notations à déchiffrer. `n^{o(1)}` désigne une fonction qui croît moins vite que toute puissance fixe de `n` ; `n^{ω(1)}`, une fonction qui croît plus vite que toute puissance fixe. Et un `s`-mouvement est un saut direct entre deux sommets d'une même face de dimension au plus `s` — le `1`-mouvement est le pas d'arête ordinaire.

Le théorème dit donc ceci : pour `n` assez grand, il existe un polytope simple à `n` facettes, en dimension plus petite que toute puissance de `n`, dont le diamètre dépasse toute puissance de `n` — et cela reste vrai si l'on autorise des sauts à travers des faces de dimension `n^{o(1)}`.

La mesure qui suit ne prétend pas approcher ces polytopes. Elle regarde trois familles élémentaires sur lesquelles la borne de Hirsch est **atteinte exactement**, pour montrer ce que l'énoncé dit des petits cas avant de dire ce que le théorème lui fait.

In [17]:
print("cube de Klee–Minty (eps = 1/3) : n = 2d facettes")
for d in range(1, 6):
    pts = sommets(facettes_km(d, eps), d)
    print(f"  d = {d} : n = {2 * d} facettes | sommets = {len(pts)} | "
          f"diamètre = {diametre(graphe(pts, d))} | n - d = {2 * d - d}")

simplexe = [([F(1), F(0)], F(1)), ([F(0), F(1)], F(1)), ([F(-1), F(-1)], F(0))]
pts = sommets(simplexe, 2)
print(f"simplexe de dimension 2 : n = 3 facettes | sommets = {len(pts)} | "
      f"diamètre = {diametre(graphe(pts, 2))} | n - d = {3 - 2}")

prisme = [([F(-1), F(0), F(0)], F(0)), ([F(0), F(-1), F(0)], F(0)),
          ([F(1), F(1), F(0)], F(1)), ([F(0), F(0), F(-1)], F(0)),
          ([F(0), F(0), F(1)], F(1))]
pts = sommets(prisme, 3)
adj = graphe(pts, 3)
print(f"prisme triangulaire : n = 5 facettes | sommets = {len(pts)} | "
      f"degrés = {sorted({len(a) for a in adj.values()})} | "
      f"diamètre = {diametre(adj)} | n - d = {5 - 3}")

cube de Klee–Minty (eps = 1/3) : n = 2d facettes
  d = 1 : n = 2 facettes | sommets = 2 | diamètre = 1 | n - d = 1
  d = 2 : n = 4 facettes | sommets = 4 | diamètre = 2 | n - d = 2
  d = 3 : n = 6 facettes | sommets = 8 | diamètre = 3 | n - d = 3


  d = 4 : n = 8 facettes | sommets = 16 | diamètre = 4 | n - d = 4
  d = 5 : n = 10 facettes | sommets = 32 | diamètre = 5 | n - d = 5
simplexe de dimension 2 : n = 3 facettes | sommets = 3 | diamètre = 1 | n - d = 1
prisme triangulaire : n = 5 facettes | sommets = 6 | degrés = [3] | diamètre = 2 | n - d = 2


### Lecture du résultat

| Famille | Facettes `n` | Dimension `d` | Sommets | Diamètre | `n - d` |
|---|---|---|---|---|---|
| cube de Klee–Minty | `2d` | `d` | `2`, `4`, `8`, `16`, `32` | `1`, `2`, `3`, `4`, `5` | `1`, `2`, `3`, `4`, `5` |
| simplexe | 3 | 2 | 3 | 1 | 1 |
| prisme triangulaire | 5 | 3 | 6 | 2 | 2 |

Sur les trois familles, le diamètre vaut exactement `n - d`. La borne de Hirsch n'y est pas seulement respectée : elle est serrée. Le cas du cube n'est pas anodin — le papier compagnon rappelle que « Klee and Walkup reduced the linear conjecture to the case n = 2d », précisément la situation du cube. Le prisme, lui, est `3`-régulier : ses six sommets ont tous trois voisins, comme l'exige la simplicité (un sommet porte `d` facettes, donc `d` arêtes).

**Ce que la section 6 disait, et ce qu'elle ne disait pas.** Sur le cube cumulé, la règle de Dantzig a pris `2^d - 1` pivots alors que l'origine et l'optimum étaient à une arête l'un de l'autre : le graphe était court, et c'est la **règle** qui allongeait le chemin. Sur les polytopes du théorème 1.1, c'est l'inverse : aucune règle ne peut plus rien y faire, puisque même le plus court chemin est long. Le papier compagnon le dit en une phrase : « Starting from a feasible vertex, a pivot can move to an adjacent vertex. Even complete knowledge of the destination cannot shorten a walk below the graph distance. »

**Une conséquence à lire comme un raisonnement, non comme une citation.** Le théorème 1.1 dit plus que le diamètre. Sur le même polytope, toute paire de sommets dont les valeurs de l'objectif diffèrent d'au moins `n^{-α}` fois l'étendue de l'objectif est à distance superpolynomiale, pour tout `α` fixé entre `0` et `1`. Un chemin qui suit les arêtes depuis n'importe quel sommet éloigné de l'optimum de cette fraction est donc long lui aussi. Tout algorithme qui va de sommet adjacent en sommet adjacent doit au moins emprunter ce chemin : aucune règle de pivot ne peut être fortement polynomiale sur toutes les entrées. Le papier compagnon rend la chose encore plus concrète (corollaire 14.3) : il donne des programmes de packing sur lesquels « the origin and the all-ones vector are the unique minimizer and maximizer of the sum of the coordinates, yet every edge path between these explicit endpoints has superpolynomial length ».

**Réfuter Hirsch n'est pas répondre à Smale.** La conjecture polynomiale parle du graphe ; la question de Smale parle d'un algorithme. Le même paragraphe les sépare explicitement : « General LP algorithms have other choices. In the companion paper [10], a strongly polynomial algorithm finds an optimum by recovering its primal and dual certificates from the inequalities. » Les deux papiers sont compagnons au sens précis : l'un ferme la route des arêtes, l'autre en ouvre une autre. Sur les instances mêmes où cette route est sans espoir, la recherche des certificats reste fortement polynomiale.

Le papier principal signale une troisième voie, intermédiaire : « A circuit walk can leave the edge graph by taking a maximal feasible step in a circuit direction. Natura constructs polynomial-length monotone circuit walks between specified vertices [108]. »

**Portée de la mesure.** Trois familles élémentaires, de dimension au plus `5`, où la borne est atteinte ; le théorème concerne des polytopes « for every sufficiently large n », dont la construction n'est pas reproduite ici. La mesure montre ce que l'énoncé dit des petits cas. Elle ne dit rien des grands.

## 12. La même garantie ailleurs : les jeux à gain moyen

La question de Smale porte sur la programmation linéaire, mais la portée d'un algorithme fortement polynomial se mesure aussi à ce qu'il entraîne. Le papier étend la garantie aux jeux sur graphes, où deux joueurs se disputent la moyenne des gains d'une marche :

> Theorem 11 (Values and optimal policies). For discounted and priority mean-payoff games on finite turn-based arenas of perfect information, there is a deterministic algorithm that computes the exact value vector v and a pair of policies. From every initial state i, the Max policy guarantees expected payoff at least v_i and the Min policy guarantees at most v_i against every strategy of the opposing player [...].

**Le cadre, dans les termes du papier.** Une arène est un graphe orienté dont chaque état appartient à l'un des deux joueurs : « Each of its n ≥ 1 states belongs to Max or Min, whose turn it is to choose an action there. » Chaque état a au moins une action. « A policy is a pure positional strategy, specifying one action at each state owned by that player. » Une partie est la suite de ses issues `e_1, e_2, …`, et son gain moyen ordinaire est « lim inf_{N→∞} (1/N) Σ_{k=1}^{N} r_{e_k} ».

L'arène est **déterministe** quand chaque action n'a qu'une issue possible. C'est le cas qui nous intéresse, et le papier le distingue nettement :

> In a deterministic arena there is one outcome per action, so the two-player algorithm is strongly polynomial, including for arbitrary real rewards.

Le théorème garantit trois choses ensemble : l'algorithme calcule le **vecteur des valeurs** (une valeur par état de départ), il rend une **politique pour chacun** des deux joueurs, et ces politiques sont **optimales** — celle de Max garantit au moins `v_i`, celle de Min au plus `v_i`, contre toute stratégie adverse.

La mesure qui suit ne reproduit pas l'algorithme. Elle construit une arène déterministe à deux états, calcule par énumération exhaustive le vecteur des valeurs et les politiques optimales, et sert à rendre concrets les trois objets du théorème : valeur, politique, garantie.

In [18]:
def gain_moyen(actions: dict, depart: int) -> F:
    """Gain moyen de la partie jouee depuis depart.
    actions[e] = (suivant, gain) : le coup joue dans l'etat e. Comme chaque etat
    n'a qu'un successeur, la partie finit dans un cycle : on moyenne ses gains."""
    vus, suite, courant = {}, [], depart
    while courant not in vus:
        vus[courant] = len(suite)
        suite.append(courant)
        courant = actions[courant][0]
    cycle = suite[vus[courant]:]
    return sum(actions[e][1] for e in cycle) / len(cycle)


def nommer(coup: tuple) -> str:
    """Description lisible d'un coup (suivant, gain)."""
    return f"vers {coup[0]} (gain {coup[1]})"


MAX, MIN = 0, 1
actions_max = {MAX: [(MIN, F(0)), (MAX, F(1, 2))]}
actions_min = {MIN: [(MAX, F(1)), (MIN, F(0))]}

print("arène déterministe à deux états : Max joue en 0, Min joue en 1")
print("  coups de Max en 0 :", " ; ".join(nommer(c) for c in actions_max[MAX]))
print("  coups de Min en 1 :", " ; ".join(nommer(c) for c in actions_min[MIN]))
print()
print("les quatre paires de politiques pures : gain moyen depuis chaque état")
gains = {}
for coup_max in actions_max[MAX]:
    for coup_min in actions_min[MIN]:
        actions = {MAX: coup_max, MIN: coup_min}
        gains[(coup_max, coup_min)] = [gain_moyen(actions, e) for e in (MAX, MIN)]
        print(f"  Max {nommer(coup_max)} | Min {nommer(coup_min)} |"
              f" depuis 0 : {gains[(coup_max, coup_min)][0]}"
              f", depuis 1 : {gains[(coup_max, coup_min)][1]}")

print()
print("état 0 : Max choisit son coup, Min répond au pire")
garantie_max = {}
for coup_max in actions_max[MAX]:
    garantie_max[coup_max] = min(gains[(coup_max, coup_min)][0] for coup_min in actions_min[MIN])
    print(f"  si Max {nommer(coup_max)} : gain garanti {garantie_max[coup_max]}")
politique_max = max(garantie_max, key=garantie_max.get)
valeur_0 = garantie_max[politique_max]

print("état 1 : Min choisit son coup, Max répond au pire")
garantie_min = {}
for coup_min in actions_min[MIN]:
    garantie_min[coup_min] = max(gains[(coup_max, coup_min)][1] for coup_max in actions_max[MAX])
    print(f"  si Min {nommer(coup_min)} : gain garanti {garantie_min[coup_min]}")
politique_min = min(garantie_min, key=garantie_min.get)
valeur_1 = garantie_min[politique_min]

print()
print(f"vecteur des valeurs : depuis 0 : {valeur_0} ; depuis 1 : {valeur_1}")
print(f"politiques optimales : Max {nommer(politique_max)} ; Min {nommer(politique_min)}")
contre_min = [gains[(politique_max, coup_min)][0] for coup_min in actions_min[MIN]]
contre_max = [gains[(coup_max, politique_min)][1] for coup_max in actions_max[MAX]]
print(f"politique optimale de Max, contre chaque réponse de Min : {[str(g) for g in contre_min]}"
      f" (tous >= {valeur_0} ? {all(g >= valeur_0 for g in contre_min)})")
print(f"politique optimale de Min, contre chaque réponse de Max : {[str(g) for g in contre_max]}"
      f" (tous <= {valeur_1} ? {all(g <= valeur_1 for g in contre_max)})")

arène déterministe à deux états : Max joue en 0, Min joue en 1


  coups de Max en 0 : vers 1 (gain 0) ; vers 0 (gain 1/2)
  coups de Min en 1 : vers 0 (gain 1) ; vers 1 (gain 0)

les quatre paires de politiques pures : gain moyen depuis chaque état
  Max vers 1 (gain 0) | Min vers 0 (gain 1) | depuis 0 : 1/2, depuis 1 : 1/2
  Max vers 1 (gain 0) | Min vers 1 (gain 0) | depuis 0 : 0, depuis 1 : 0
  Max vers 0 (gain 1/2) | Min vers 0 (gain 1) | depuis 0 : 1/2, depuis 1 : 1/2
  Max vers 0 (gain 1/2) | Min vers 1 (gain 0) | depuis 0 : 1/2, depuis 1 : 0

état 0 : Max choisit son coup, Min répond au pire
  si Max vers 1 (gain 0) : gain garanti 0
  si Max vers 0 (gain 1/2) : gain garanti 1/2
état 1 : Min choisit son coup, Max répond au pire
  si Min vers 0 (gain 1) : gain garanti 1/2
  si Min vers 1 (gain 0) : gain garanti 0

vecteur des valeurs : depuis 0 : 1/2 ; depuis 1 : 0
politiques optimales : Max vers 0 (gain 1/2) ; Min vers 1 (gain 0)
politique optimale de Max, contre chaque réponse de Min : ['1/2', '1/2'] (tous >= 1/2 ? True)
politique optimale 

### Lecture du résultat

Le tableau des quatre paires se lit ainsi. Depuis l'état 0, trois paires sur quatre donnent `1/2` ; seule la paire où Max va vers 1 et Min reste en 1 donne `0`. Depuis l'état 1, deux paires donnent `1/2` et deux donnent `0`. Ce n'est pas un calcul d'équilibre : c'est l'énumération exhaustive de tout ce que des politiques pures positionnelles peuvent produire sur cette arène.

**Valeur.** En l'état 0, Max choisit : `0` s'il va vers 1, `1/2` s'il reste en 0. Le maximum vaut `1/2`, atteint par le coup que la sortie étiquette « vers 0 ». En l'état 1, Min choisit : `1/2` s'il va vers 0, `0` s'il reste en 1. Le minimum vaut `0`. Le vecteur des valeurs est donc `(1/2, 0)` — exactement l'objet que le théorème 11 appelle « the exact value vector v ».

**Politiques et garantie.** La politique optimale de Max est de rester en 0 ; les deux dernières lignes vérifient la garantie dans les termes mêmes du théorème : contre **chaque** réponse de Min, le gain vaut `1/2`, donc au moins `v_0` ; contre **chaque** réponse de Max, la politique de Min rend `0`, donc au plus `v_1`. La politique de Min ne prétend pas gagner : elle empêche Max de gagner davantage.

**Ce que la mesure est, et ce qu'elle n'est pas.** Une arène à deux états et quatre actions vérifie les définitions — partie, gain moyen comme moyenne de cycle, politique pure positionnelle, valeur, garantie — et rien de plus. Elle ne mesure pas l'algorithme, qui n'est pas reproduit. Le théorème 11 est bien plus fort : « In a deterministic arena there is one outcome per action, so the two-player algorithm is strongly polynomial, including for arbitrary real rewards. » Les gains de cette arène sont rationnels et petits ; l'énoncé les autorise **réels arbitraires**, et c'est précisément la partie qu'aucun calcul en virgule flottante ne démontrerait.

**Pourquoi ce théorème est dans ce carnet.** Parce que la même machinerie y entre. Le papier décrit la composition de la preuve : « A fixed-point computation proposes actions for the two players, and linear programming determines what each proposed policy guarantees against a best response. » Évaluer une politique contre la meilleure réponse adverse est un problème à un joueur : un programme linéaire. C'est là que l'algorithme fortement polynomial du théorème 1 est appelé, et c'est ce qui étend sa garantie aux jeux. Le papier en tire deux conséquences qu'il énonce lui-même : dans un processus de décision markovien, « the step bound is polynomial in the size of the arena and its discrete data even for arbitrary real transition probabilities supplied as scalars » ; et, en rendant un état cible absorbant, un algorithme en temps polynomial pour les jeux stochastiques simples, « whose threshold problem Condon placed in NP ∩ coNP ». Ce carnet n'en dit pas davantage : il rapporte ce que le papier écrit, sans reconstituer l'algorithme.

## 13. Ce que le papier énonce, ce que le Lean énonce

« Résultat formalisé » est une phrase facile à écrire et coûteuse à vérifier. La vérification consiste à mettre côte à côte deux textes : l'énoncé du papier, et l'énoncé que la bibliothèque Lean déclare prouver. C'est ce que fait cette section. Tout y est **lu**, ligne par ligne, dans deux corpus — le texte des papiers, et l'arbre Lean qui les accompagne —, sauf une cellule de calcul, qui reprend une instance déjà mesurée pour estimer la portée d'un écart.

Ce que l'archive contient, vérifié par lecture : le développement `lean/OAI/Optimization/LinearProgramming/` et `lean/OAI/Geometry/Hirsch/`, tous deux en fichiers `.lean`, un répertoire `lean/ComparatorChallenges/` d'énoncés autonomes, et un fichier `overview.tex` à la racine. Le répertoire `build/` n'y existe pas : l'archive ne contient aucun artefact compilé, et aucun `lake build` n'a été lancé pour écrire cette section. Les énoncés ci-dessous sont lus, pas re-prouvés — leur compilation est le fait de la chaîne d'intégration continue du dépôt qui les héberge, que ce carnet ne rejoue pas.

Voici l'énoncé du papier principal, dans sa version anglaise et mot pour mot :

> **Theorem 1 (Strongly polynomial linear programming).** For arbitrary real `A`, `b`, `c`, a deterministic strongly polynomial algorithm returns one of the following certificates for `max{c⊤x : Ax ≤ b}`.
>
> 1. **Finite optimum.** A strictly complementary primal–dual pair `x`, `y` with `s = b −Ax ≥0`, `y ≥0`, `A⊤y = c`, `s⊤y = 0`, `s + y > 0`.
> 2. **Unbounded objective.** A feasible point `x` and an improving ray `r` with `Ar ≤0` and `c⊤r > 0`.
> 3. **Infeasibility.** A vector `y ≥0` with `A⊤y = 0`, `b⊤y < 0`, and `|supp y| ≤1 + rank A`.

Le paragraphe qui suit ajoute une phrase : « The arithmetic result extends to every ordered field. »

Trois traits de cet énoncé méritent d'être soulignés, parce que la comparaison va porter sur eux. Les données sont **réelles arbitraires** — pas rationnelles, pas binaires. La sortie du cas 1 est un couple primal–dual **strictement** complémentaire : `s⊤y = 0` dit qu'à chaque contrainte, l'écart ou le multiplicateur est nul, et `s + y > 0` ajoute qu'ils ne sont jamais nuls **tous les deux**. Et le cas 3 joint à l'existence du certificat une borne de support — celle que la section 5 a vérifiée sur deux familles entières.

### L'énoncé Lean, traduit

La formalisation est écrite sur une machine à registres, et non sur un algorithme abstrait. Le théorème central, `StronglyPolynomialOptimization`, se lit ainsi : il existe un programme `p` et quatre constantes `c`, `e`, `bC`, `bE` — choisies **avant** toute entrée — telles que, pour toutes dimensions `m`, `d` non nulles et toutes données `A`, `b`, `obj` **rationnelles**, il existe un nombre de pas `k` et une exécution `t` de `p` depuis l'entrée encodée pour laquelle

- `k ≤ polyBound c e (m + d)` — le nombre de pas est borné par un polynôme en les **dimensions seules** ;
- `∀ i, scalarBound (polyBound bC bE (totalBits input)) (t i)` — tout scalaire présent en mémoire garde une taille binaire polynomiale en la taille encodée de l'entrée ;
- `OptimizationOutput A b obj (t (Fin.last k))` — le dernier état porte l'un des trois certificats.

avec `polyBound c e n = c * (n + 1) ^ e`. Les deux premiers conjoints sont exactement les deux quantités mesurées en section 9 : un nombre de pas qui ne dépend pas de la taille des coefficients, et des scalaires intermédiaires qui ne gonflent pas. C'est cela, le contenu formel de « fortement polynomial ».

Les déclarations qui portent le résultat, avec leur emplacement dans l'archive :

| Emplacement sous `lean/` | Ligne | Déclaration | Ce qu'elle énonce |
|---|---|---|---|
| `OAI/Optimization/LinearProgramming/Specification.lean` | 16 | `lpInput` | la bande d'entrée : `A` ligne par ligne, puis `b` ; les dimensions dans les registres 0 et 1 |
| idem | 21 | `Feasible` | `A *ᵥ x ≤ b` |
| idem | 24 | `Certificate` | `0 ≤ y`, `Aᵀy = 0`, `b · y < 0`, et `card {i | y i ≠ 0} ≤ A.rank + 1` |
| idem | 30 | `FeasibilityOutput` | étiquette 0 : un point réalisable ; étiquette 1 : un certificat de Farkas |
| idem | 36 | `NormalizedFeasibilityOutput` | idem, avec de plus `b · y = -1` |
| idem | 43 | `OptimizationOutput` | étiquettes 0, 1, 2 : infaisabilité, optimum primal–dual, base réalisable et rayon normalisé |
| idem | 55 | `StronglyPolynomialFeasibility` | un programme fixe décide la réalisabilité, pas et scalaires bornés |
| idem | 65 | `StronglyPolynomialOptimization` | même chose pour l'optimisation, objectif `obj` compris |
| idem | 76 | `StronglyPolynomialOrderedFieldFeasibility` | le même programme sur **tout corps ordonné** `K`, pour la réalisabilité |
| `OAI/Optimization/LinearProgramming/RAM/Basic.lean` | 91 | `polyBound` | `c * (n + 1) ^ e` |
| `OAI/Optimization/LinearProgramming/Feasibility.lean` | 13 | `strongly_polynomial_feasibility` | la preuve du théorème de réalisabilité |
| `OAI/Optimization/LinearProgramming/Optimization.lean` | 13 | `strongly_polynomial_optimization` | la preuve du théorème d'optimisation |
| `OAI/Optimization/LinearProgramming/Farkas/Basic.lean` | 94, 137, 147 | `alternative`, `certificate_excludes_feasible`, `sparse_alternative` | l'alternative de Farkas, sa contraposée, et sa version à support borné |
| `OAI/Geometry/Hirsch/Main.lean` | 50, 66, 80, 106, 111 | `theorem_1_1`, `superpolynomial_graph_diameter`, `graph_diameter_beats_polynomials`, `not_polynomialHirsch`, `theorem_1_2` | l'énoncé du papier compagnon et ses conséquences |

Les deux théorèmes tiennent chacun en quelques lignes : le premier appelle `NormalSubspace.normal_decision` puis `Recovery.Principal.feasibility_of_subspace` ; le second compose le premier avec `PrimalDual.stronglyPolynomialOptimization_of_feasibility`. Autrement dit, la formalisation factorise l'optimisation en un appel à la réalisabilité — la documentation de la déclaration le dit : « The separate reduction in §2 bounds the number of feasibility calls. »



In [19]:
A_seg = [[1, 0], [1, 1], [1, -1]]           # l'instance degeneree de la section 3
b_seg = [1, 1, 1]
c_seg = [1, 0]
x_seg = [F(1), F(0)]                        # son optimum, de valeur 1
s_seg = [F(b_seg[i]) - sum(F(A_seg[i][j]) * x_seg[j] for j in range(2)) for i in range(3)]
valeur = sum(F(ci) * xi for ci, xi in zip(c_seg, x_seg))
print("instance de la section 3, sous la forme du théorème : x =", fmt(x_seg),
      "| s =", fmt(s_seg), "| c.x =", valeur)
for t in (F(0), F(1, 4), F(1, 2)):
    y = [1 - 2 * t, t, t]                   # A^T y = c impose y = (1 - 2t, t, t)
    ATy = [sum(F(A_seg[i][j]) * y[i] for i in range(3)) for j in range(2)]
    bdy = sum(F(b_seg[i]) * y[i] for i in range(3))
    print(f"t = {t} : y = {fmt(y)} | y >= 0 : {all(v >= 0 for v in y)} | "
          f"A^T y = c : {ATy == [F(v) for v in c_seg]} | b.y = c.x : {bdy == valeur} | "
          f"s.y = 0 : {sum(si * yi for si, yi in zip(s_seg, y)) == 0} | "
          f"s + y > 0 : {all(si + yi > 0 for si, yi in zip(s_seg, y))}")


instance de la section 3, sous la forme du théorème : x = (1, 0) | s = (0, 0, 0) | c.x = 1
t = 0 : y = (1, 0, 0) | y >= 0 : True | A^T y = c : True | b.y = c.x : True | s.y = 0 : True | s + y > 0 : False
t = 1/4 : y = (1/2, 1/4, 1/4) | y >= 0 : True | A^T y = c : True | b.y = c.x : True | s.y = 0 : True | s + y > 0 : True
t = 1/2 : y = (0, 1/2, 1/2) | y >= 0 : True | A^T y = c : True | b.y = c.x : True | s.y = 0 : True | s + y > 0 : False


### Les écarts, un par un

Le tableau ci-dessus aligne les déclarations ; il ne dit pas si elles disent la même chose. Voici les six points où les deux textes diffèrent, chacun avec la mesure qui l'établit.

| # | Le papier | L'archive Lean | Mesure |
|---|---|---|---|
| 1 | données **réelles** arbitraires, puis « extends to every ordered field » | l'optimisation est énoncée sur `ℚ` seulement : `A : Matrix (Fin m) (Fin d) ℚ`, `obj : Fin d → ℚ` | le corps ordonné n'apparaît que dans `StronglyPolynomialOrderedFieldFeasibility` (l. 76), pour la **réalisabilité** et avec le certificat normalisé ; le nom de cette déclaration n'apparaît qu'une fois dans tout l'arbre, à sa définition — aucun théorème ne la conclut sous ce nom |
| 2 | cas 1 : couple **strictement** complémentaire, `s + y > 0` | étiquette 1 de `OptimizationOutput` : réalisable, `0 ≤ y`, `Aᵀy = c`, `c·x = value`, `b·y = value`, optimal contre tout `z` réalisable | aucune clause `s + y > 0` dans l'étiquette 1 : sur ce point, l'énoncé formalisé est **plus faible** que celui du papier — la cellule précédente mesure l'écart, sur l'instance de la section 3 |
| 3 | cas 2 : `c⊤r > 0` | étiquette 2 : `A v ≤ 0` et `c·v = 1` | normalisation d'un rayon, pas un écart de contenu : un rayon se remet à l'échelle |
| 4 | cas 3 : `b⊤y < 0` | `Certificate` : `b · y < 0` ; `sparse_alternative` et `NormalizedFeasibilityOutput` : `b · y = -1` | les deux formes coexistent ; le certificat rendu en section 5 vaut `b·y = -1/2`, conforme au `< 0` et à `Certificate`, pas à la normalisation `-1` |
| 5 | « Theorem 1 » | en-têtes : « Theorem 1.1 » (`Feasibility.lean`, l. 4), « Corollary 1.2 » (`Optimization.lean`, l. 4), « Theorem 8.1 » (`Specification.lean`, l. 5) | dans le texte du papier : `Theorem 1.1` zéro occurrence, `Corollary 1.2` zéro, `Theorem 8.1` zéro ; le papier numérote **à plat** (`Theorem 1`, … , `Theorem 23`, `Corollary 6`, `7`, `9`, `12`, `Proposition 10`) |
| 6 | le texte du papier ne contient ni « Lean » ni « formaliz » | `OAI/Optimization` et `OAI/Geometry` : zéro occurrence du mot `sorry` (et de `sorryAx`, `by admit`, `· admit`, `exact admit`) ; tout `OAI/` : sept occurrences, toutes en commentaires | les `sorry` visibles sont dans `ComparatorChallenges/` : `LinearProgramming.lean` l. 167 et 170 réénoncent les deux théorèmes en `:= by sorry`, `Hirsch.lean` en porte six — un répertoire d'énoncés autonomes, pas le développement |

Le point 5 mérite un mot de plus, parce qu'il se lit dans les deux sens. Le papier **compagnon**, lui, numérote hiérarchiquement — `Theorem 1.1` y apparaît cinq fois, `Theorem 1.2` trois fois, `Corollary 14.3` deux fois — et les fichiers Lean de Hirsch s'y conforment : `theorem_1_1`, `theorem_1_2`, `not_polynomialHirsch`. Deux conventions de numérotation coexistent donc dans l'archive, et celle des en-têtes du papier principal ne se retrouve pas dans son texte. La conséquence est pratique : la correspondance entre une déclaration Lean et un énoncé du papier s'établit **par le contenu**, jamais par un numéro.

Deux mots de forme avant d'en venir au point 2. Le simplexe de la section 2 suppose `x >= 0`, et le dual qu'il manipule s'écrit avec des inégalités : `Aᵀy >= c`, avec des coûts réduits positifs ou nuls. C'est pourquoi la face duale optimale de la section 3 était un triangle. Les deux textes comparés ici prennent l'autre forme, `max{c⊤x : Ax ≤ b}`, sans contrainte de signe sur `x` : `Feasible` est défini par `A *ᵥ x ≤ b`, et rien d'autre. Le dual y impose l'égalité `Aᵀy = c`.

Le point 2 est le seul qui touche au contenu mathématique, et la cellule précédente le mesure sur l'instance dégénérée de la section 3, replacée dans cette forme. L'optimum n'y change pas : `x = (1, 0)`, pour la valeur `1`. Les deux équations du dual, `y1 + y2 + y3 = 1` et `y2 − y3 = 0`, laissent un degré de liberté : `y = (1 − 2t, t, t)`, avec `t` de `0` à `1/2` pour que `y` reste positif. La face duale est ici un segment, et la cellule en prend trois points.

La sortie sépare les deux énoncés. Les trois points vérifient `y >= 0`, `A^T y = c`, `b.y = c.x` et `s.y = 0` : ce sont trois sorties admissibles pour l'étiquette 1 de `OptimizationOutput` — l'optimalité contre tout `z` réalisable découle de `b.y = c.x`, puisque `x` est réalisable et `y` positif. Mais `s + y > 0` n'est vrai qu'au point intérieur, `t = 1/4`. Aux deux extrémités, `t = 0` et `t = 1/2`, une coordonnée de `y` est nulle alors que l'écart de la contrainte correspondante l'est aussi : `s` est nul partout. L'énoncé Lean accepte les trois ; celui du papier n'accepte que `y = (1/2, 1/4, 1/4)`.

Ce que cela veut dire, et ce que cela ne veut pas dire. Cela ne dit rien contre l'optimalité : les trois points sont optimaux, et `b.y = c.x` le certifie. Cela ne dit rien contre la dualité forte, ni contre les bornes de pas et de taille des scalaires. Cela dit qu'un algorithme qui ne rendrait que des extrémités de la face duale satisferait la version formalisée **sans** satisfaire celle du papier — et ce carnet ne prétend pas que l'algorithme formalisé se comporte ainsi : il constate que son énoncé ne l'exclut pas. La complémentarité stricte est une propriété supplémentaire que le papier demande à la sortie, que la formalisation n'énonce pas.

### Portée de cette section

Rien n'a été compilé ici : les lignes citées sont lisibles dans l'archive, dont le chemin est donné en fin de carnet, et leur compilation relève de la chaîne d'intégration du dépôt qui les héberge. La lecture porte sur les énoncés, pas sur les preuves : savoir si `feasibility_of_subspace` établit vraiment le théorème demanderait de suivre la chaîne des modules, ce que cette section ne fait pas. Elle établit trois choses, et pas davantage : les deux textes ne sont pas identiques ; l'écart est localisé (champ, complémentarité stricte, normalisations) ; la numérotation ne permet pas de les apparier, seul le contenu le permet.


## 14. Mesuré, cité, ouvert

La convention de lecture annoncée en tête de ce carnet a été suivie section par section. Elle se récapitule en trois listes : ce que les cellules ont produit, ce que les sources affirment, et ce que ni l'une ni l'autre n'établit.

### Mesuré dans ce carnet

Toutes les valeurs ci-dessous figurent dans les sorties des cellules. Elles sont en arithmétique rationnelle exacte, sauf les deux pentes de la section 8, ajustées sur des flottants.

| La question | Section | Le résultat |
|---|---|---|
| Le simplexe exact rend-il l'optimum du socle ? | 2 | `x = (2, 1)`, valeur `8`, en `2` pivots ; le témoin PuLP/CBC rend le même optimum |
| Un optimum dégénéré admet-il un couple strictement complémentaire ? | 3 | le couple du simplexe n'en est pas un (`s + y > 0` faux) ; la moyenne des `3` sommets de la face duale, `y = (1/3, 1/2, 1/6)`, en est un |
| La non-bornitude a-t-elle un certificat ? | 4 | le rayon `r = (1, 1)`, avec `A r ≤ 0` et `c·r = 2` |
| La borne de support d'un certificat de Farkas tient-elle ? | 5 | `729` puis `19683` systèmes, `0` certificat manquant ; support au plus `rang + 1`, borne atteinte |
| Le simplexe peut-il être exponentiel en la dimension ? | 6 | `2^d - 1` pivots sous Dantzig, jusqu'à `1023` en dimension `10` ; `3193` sous Bland en dimension `16` |
| Deux programmes de mêmes dimensions peuvent-ils différer par la longueur de leurs coefficients ? | 7 | de `11` à `161` bits pour le plus grand coefficient, à `(m, n) = (3, 2)` fixés |
| L'ellipsoïde dépend-il de cette longueur ? | 8 | `5.083` coupes par bit de `R`, `2.676` par bit de `L`, contre `5.299` et `2.649` prédits par la loi de volume |
| Que pèse un sommet d'un polyèdre à coefficients de `L` bits ? | 9 | `2L` bits, de `20` à `400` bits sur la famille mesurée, sous la borne `2L + 1` |
| Un échelonnage de colonnes répare-t-il le déséquilibre de circuits ? | 10 | non sur les `5` échelonnages essayés : jamais sous `1/u = 10/3` |
| La borne de Hirsch est-elle serrée ? | 11 | le diamètre vaut `n - d` sur les trois familles construites |
| Que mesure une arène à deux états ? | 12 | valeurs `(1/2, 0)`, et deux politiques dont la garantie tient contre toute réponse |
| L'énoncé Lean est-il celui du papier ? | 13 | trois sorties admissibles pour l'étiquette 1 de `OptimizationOutput`, une seule pour le théorème 1 |

### Cité, non mesuré

- **Le théorème 1** : l'existence d'un algorithme fortement polynomial, ses trois sorties, la taille polynomiale des nombres intermédiaires. Aucune cellule ne l'implémente ni ne compte ses pas.
- **Le théorème 11** et ses deux conséquences, sur les processus de décision markoviens et sur les jeux stochastiques simples : rapportés depuis le texte.
- **Le papier compagnon** : son théorème 1.1, les exemples de Santos puis de Matschke, Santos et Weibel, son corollaire 14.3. Aucun de ces polytopes n'est construit ici.
- **Les énoncés Lean** : lus dans les fichiers du dépôt `openai/math`, avec leurs numéros de ligne.
- **Les questions posées avant la réponse** : le neuvième problème de Smale et la règle de pivot de la note de Klee et Minty.

### Ouvert

- **L'algorithme.** Ce carnet mesure des obstacles et contrôle des certificats ; il n'implémente pas celui du théorème 1, et ne décide donc la réalisabilité d'aucun programme en un nombre de pas borné par un polynôme en `m` et `n` seuls.
- **L'exposant.** Ce carnet n'a pas trouvé, dans le texte extrait du manuscrit, de borne numérique explicite de la forme `(m + n)^k` ; l'énoncé Lean, `polyBound c e (m + d)`, nomme des constantes dont la valeur n'est pas donnée.
- **Les preuves.** L'absence de `sorry` dans les deux répertoires du développement se mesure en lisant les fichiers ; elle ne dit rien de leur compilation, qui n'a pas été rejouée ici.
- **Les données réelles arbitraires.** Tout est rationnel ou flottant dans ces cellules. Le cas des réels, qui est celui du théorème, n'y est pas testé.
- **La complémentarité stricte côté Lean.** L'étiquette 1 de `OptimizationOutput` ne l'exige pas, et la mesure de la section 13 ne dit rien de ce que produit l'algorithme formalisé.
- **Les plages.** La récurrence de Bland est observée jusqu'à la dimension `16`, l'échelonnage sur `5` choix, l'alternative de Farkas sur `3` contraintes et des coefficients dans `{-1, 0, 1}`, la borne de Hirsch sur des dimensions au plus `5`.
- **La note de 1972.** C'est un scan : son texte courant a pu être lu, ses inégalités affichées non. La famille exécutée en section 6 est une variante à coefficients cumulés, décrite dans ce carnet.

## Conclusion — deux tailles, et une seule était bornée

Le [07 — Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) demandait ce que mesure exactement la taille d'une entrée. Sur un programme linéaire, la réponse tient en deux quantités, et le neuvième problème de Smale demande si la seconde suffit à elle seule :

| Taille | Ce qu'elle compte | Ce que les cellules mesurent | Ce que le théorème 1 garantit (cité) |
|---|---|---|---|
| la **longueur binaire** de l'entrée | les bits des coefficients | les coupes de l'ellipsoïde croissent avec elle (section 8) ; un sommet s'écrit en `2L` bits (section 9) | des nombres intermédiaires de taille polynomiale en cette longueur |
| les **dimensions** `m` et `n` | le nombre de contraintes et de variables | `2^d - 1` pivots pour Dantzig (section 6) ; sur l'instance de la section 7, `1` pivot quelle que soit la longueur | un nombre de pas polynomial en `m, n`, pour des données réelles arbitraires |

La première ligne était acquise : « For rational coefficients, the ellipsoid method of Khachiyan [88, 89] gave an answer polynomial in the length of the input. » La seconde était ouverte. Les deux règles de pivot examinées ici ne la tiennent pas (section 6), et le théorème 1.1 du papier compagnon interdit à toute marche le long des arêtes de la tenir : il construit des polytopes simples « whose graph diameter grows faster than any polynomial in their number of facets ». Le théorème 1 l'établit par une autre route, qui va chercher les certificats primal et dual dans les inégalités plutôt que dans les arêtes du polytope.

Reste la leçon de la section 13. Entre le manuscrit et sa formalisation, le résultat a changé de forme : les données de l'optimisation y sont devenues rationnelles, et la complémentarité stricte a quitté l'énoncé. « Résultat formalisé » ne se recopie pas dans une phrase. Il se compare, énoncé contre énoncé.

## À retenir

1. **Deux tailles, pas une.** À `(m, n) = (3, 2)` fixés, le plus grand coefficient passe de `11` à `161` bits et le simplexe fait toujours `1` pivot : sur cette instance, son nombre de pas ne dépend pas de la longueur des nombres.
2. **Le simplexe n'est pas fortement polynomial.** Sur le cube cumulé, l'optimum est à une arête de l'origine ; la règle de Dantzig fait pourtant `2^d - 1` pivots, `1023` en dimension `10`, et la règle de Bland `3193` en dimension `16`.
3. **L'ellipsoïde est polynomial en la longueur, pas en les dimensions.** À `(m, n)` fixés, ses coupes suivent `L` au rythme de `2.676` coupes par bit, et `R` au rythme de `5.083` — deux pentes proches des `2.649` et `5.299` de sa loi de volume.
4. **Les sorties de l'algorithme sont des certificats, et ils se vérifient.** Sur `19683` systèmes, `0` certificat de Farkas manquant, jamais plus de `rang + 1` coordonnées non nulles ; le rayon de la section 4 se contrôle en deux produits.
5. **Le papier et le Lean ne disent pas la même chose.** Sur l'instance de la section 13, trois sorties satisfont l'étiquette 1 de `OptimizationOutput` là où le théorème 1 n'en admet qu'une.

## Vérifiez votre compréhension

1. Pourquoi la longueur binaire de l'entrée et les dimensions `(m, n)` sont-elles deux tailles différentes d'un même programme ?
2. Sur le cube cumulé, d'où vient le nombre de pivots de la règle de Dantzig : de la distance à parcourir, ou du choix des pivots ?
3. Pourquoi dit-on que l'ellipsoïde est polynomial en la longueur de l'entrée, et non en les dimensions ?
4. Le théorème 1.1 du papier compagnon réfute-t-il la conjecture de Hirsch dans son énoncé d'origine ?
5. Sur le cas d'un optimum fini, en quoi l'énoncé formalisé est-il plus faible que le théorème 1, et quelle cellule le mesure ?

<details><summary>Réponses</summary>

1. Parce que l'une peut changer sans l'autre. Le dernier bloc de la section 7 garde `(m, n) = (3, 2)` et fait passer le plus grand coefficient de `11` à `161` bits : l'entrée s'allonge, les dimensions ne bougent pas.
2. Du choix des pivots. Sur ce polytope, le diamètre vaut `d` et l'optimum est à une arête de l'origine, à toutes les dimensions mesurées ; la règle de Dantzig fait pourtant `2^d - 1` pivots, `1023` en dimension `10`. Le graphe est court, c'est le chemin suivi qui est long.
3. Parce que son comptage suit la longueur des nombres. À `(m, n)` fixés, les coupes passent de `16` à `422` quand `L` va de `10` à `160`, au rythme de `2.676` coupes par bit, proche des `2.649` de sa loi de volume — pendant que le simplexe fait `1` pivot.
4. Non. La borne d'origine, `n - d`, a été réfutée plus tôt par Santos, avec un polytope de dimension `43` et `86` facettes. Le théorème 1.1 réfute la version affaiblie, la conjecture de Hirsch polynomiale ; sur les trois familles de la section 11, la borne d'origine est au contraire atteinte exactement.
5. L'étiquette 1 de `OptimizationOutput` ne comporte pas de clause `s + y > 0`. La dernière cellule de code de la section 13 prend trois points de la face duale d'une même instance : les trois satisfont les conditions de l'étiquette 1, un seul satisfait `s + y > 0`.

</details>

## Pour aller plus loin

- **Les textes sources**, archivés au gisement partagé (chemins en tête de ce carnet) : la prépublication du 24 novembre 2026 ; le papier compagnon ; la note de Klee et Minty ; la liste de Smale.
- **La formalisation et sa notice**, dans le dépôt `openai/math` : le développement sous `lean/OAI/Optimization/LinearProgramming/` et `lean/OAI/Geometry/Hirsch/`, le fichier `overview.tex` à la racine.
- **La question d'origine** : [Complexity-01 — Compter des pas](Complexity-01-StepCounting-Python.ipynb) pour la notion de taille d'entrée ; [Complexity-07 — Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) pour ce qu'une description courte a de coûteux à vérifier ; [Complexity-07b — Le tour des grands nombres](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb) et [Complexity-07c — Le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) pour les vagues de résultats dont celui-ci est issu.
- **Le socle** : [Search-09 — Programmation linéaire et simplexe](../../Search/Part1-Foundations/Search-09-LinearProgramming.ipynb) résout avec PuLP l'instance de production reprise en section 2.
- **Prolongements laissés ouverts** : lire dans le manuscrit la section 2, celle qui explique « how the algorithm combines such observations, even when the initial system has arbitrarily large circuit imbalance », et la preuve du théorème 11 ; comparer les déclarations `sparse_alternative` et `NormalizedFeasibilityOutput` avec le certificat de la section 5.